# 📗 그래프 집계: 세고·합치고·가공하기

지난 시간까지는 그래프에서 **패턴을 찾아 값을 꺼내는** 조회를 배웠습니다. 이번 시간에는 그 값들을 **한데 모아 요약**합니다. 이 질병을 치료하는 약이 몇 개인가(개수), 평균은 몇 개인가(평균), 그 약들이 무엇인가(목록). 표 계산에서 쓰던 "합계·평균"을 그래프 위에서 하는 셈입니다.

데이터는 **의료 지식그래프**입니다. 실제 공개 데이터(Hetionet)에서 재배포가 자유로운 부분만 추린 것으로, 노드 **15,540개**·관계 **91,966개**입니다. 표로 치면 9만 행이 넘어, 지금까지 다룬 20~30개짜리 연습 그래프와는 규모가 다릅니다.

| 노드 종류 | 개수 | 무엇인가 |
|---|---|---|
| `Gene` | 13,113 | 유전자(그 산물인 단백질까지 포함해 부릅니다) |
| `Compound` | 1,531 | 약물 성분 |
| `Symptom` | 415 | 증상 |
| `PharmacologicClass` | 345 | 약효분류(작용 방식이 같은 약물 묶음) |
| `Disease` | 136 | 질병 |

관계 중 오늘 자주 쓸 네 가지는 이렇습니다.

- `(:Compound)-[:TREATS]->(:Disease)` : 그 약이 **질병 자체를 조절**한다고 보고됐다
- `(:Compound)-[:PALLIATES]->(:Disease)` : 그 약이 **증상을 눅인다**고 보고됐다
- `(:Compound)-[:BINDS]->(:Gene)` : 그 약이 그 유전자 산물에 **결합**한다(표적)
- `(:Disease)-[:PRESENTS]->(:Symptom)` : 그 질병이 그 **증상으로 나타난다**

> **`TREATS` 와 `PALLIATES` 는 다른 말입니다.** 앞은 병 자체를 다스리는 것이고 뒤는 아픈 느낌을 덜어 주는 것입니다. 세어서 하나로 합치면 의학적으로 틀린 답이 나옵니다. 오늘 6-2 에서 이 둘을 가르는 법을 배웁니다.

> 이 데이터는 2016년에 정리된 것이고, 관계는 "그렇다고 **보고된 적이 있다**"는 뜻입니다. "효과가 입증됐다"와는 다릅니다. 집계 결과를 읽을 때 이 차이를 기억하세요.

## ⏪ 복습: 지난 시간까지

- **MATCH 패턴**으로 노드·관계를 그림 그리듯 찾고, **WHERE** 로 걸러 **RETURN** 으로 꺼냈습니다.
- **가변길이 패턴**(`*1..3`)과 `shortestPath` 로 몇 칸 떨어진 곳까지 훑고, `length`·`nodes`·`relationships` 로 그 경로를 뜯어봤습니다.
- **OPTIONAL MATCH** 로 짝이 없는 것도 빠뜨리지 않고 남겼습니다.
- **리스트 표현식**(`[n IN 목록 | 표현식]`)으로 목록에서 값만 뽑고, **패턴 술어**(`WHERE (a)-[:R]->(b)`)와 `EXISTS { }` 로 행을 늘리지 않고 관계의 유무만 조건으로 걸었습니다.
- **WITH** 로 중간 결과를 다음 단계에 넘기고, **ORDER BY·SKIP·LIMIT** 로 정렬·쪽 넘기기·상위 N 을 얻었습니다.
- 오늘은 여기에 **집계 함수**(count·sum·avg·min·max·collect)를 더해 여러 행을 **한 줄로 요약**하고, `UNWIND` 로 다시 펼치는 법까지 봅니다.

**오늘의 목표**

**1. 개수 세기: count**
- [ ] (1-1) RETURN 의 **비집계 항이 곧 묶는 기준**이라는 것을 알고 항목별로 센다.
- [ ] (1-2) `count(*)` 와 `count(x)` 가 **다른 답**을 내는 자리를 가려낸다(짝이 없는 행).
- [ ] (1-3) 경로가 갈라지는 자리에서 노드 개수를 물으면 `count(DISTINCT x)` 를 쓴다.

**2. 수치 요약: min·max·avg·sum**
- [ ] (2-1) `WITH` 로 한 번 집계한 값을 **다시 집계**한다(2단계 집계).
- [ ] (2-2) 집계가 **빠진 값을 건너뛴다**는 것과 **정수끼리 나누면 버림**이 된다는 것을 확인하고,
        건너뛰는 대신 `coalesce` 로 **채웠을 때** 분모가 어떻게 달라지는지 본다.
- [ ] (2-3) `percentileCont` 로 **중앙값·분위수**를 내어 평균이 가린 분포를 본다.

**3. 리스트로 모으고 다시 펼치기: collect·UNWIND**
- [ ] (3-1) `collect` 로 값들을 **리스트 하나**로 모은다.
- [ ] (3-2) `UNWIND` 로 그 리스트를 **다시 한 줄씩** 펼친다.
- [ ] (3-3) 이미 있는 리스트를 **조건으로 걸러 쓴다**(리스트 컴프리헨션).
- [ ] (3-4) 패턴을 **바로 리스트로** 받는다(패턴 컴프리헨션).

**4. 집계한 뒤 거르기: WITH + WHERE**
- [ ] (4-1) 집계 결과에 대한 조건은 **`WITH` 뒤 `WHERE`** 에 건다(SQL 의 HAVING 에 해당).
- [ ] (4-2) 같은 답을 `COUNT { }` 서브쿼리로 짧게 내고, 어느 쪽을 쓸지 가른다.

**5. 집계 결과를 저장: SET 파생 속성**
- [ ] (5-1) `SET` 으로 집계 결과를 **파생 속성**으로 저장한다.
- [ ] (5-2) 저장한 값은 **낡는다**는 것을 알고 `REMOVE` 로 정리한다.

**6. 조건별로 값 만들기: CASE**
- [ ] (6-1) 조건형 `CASE` 로 **구간 라벨**을 만들고, 그 라벨을 묶는 기준으로 삼는다.
- [ ] (6-2) 값이 몇 가지로 정해져 있을 때는 **단순형** `CASE` 를 쓴다.
- [ ] (6-3) 집계 함수 **안에서** `CASE` 로 행마다 점수를 매긴다.

**7. 내장 함수: 외우지 말고 찾아 쓰기**
- [ ] (7-1) `SHOW FUNCTIONS` 로 **내 데이터베이스의 함수를 직접 훑는다**.
- [ ] (7-2) 갈래별로 하나씩 써 보고, `sum` 과 `reduce` 가 **접는 대상**이 다르다는 것을 안다.
- [ ] (7-3) 공식 문서에서 **빠진 값 처리**까지 확인하며 찾는 법을 익힌다.

오늘 조회할 그래프의 전체 모습입니다. 다섯 종류의 노드가 열두 종류의 관계로 이어져 있습니다. 데모는 **약물-질병 축**, 따라하기는 **질병-증상 축**으로 갈라 같은 기술을 두 번 연습합니다.

<img src="images/그래프_한눈에_의료.png" width="820">

다섯 종류의 노드는 각각 이런 뜻입니다. 이름(레이블)은 데이터 그대로 영어입니다.

| 레이블 | 한글 이름 | 설명 | 예 |
|---|---|---|---|
| `Compound` | 약물 | 약의 유효 성분 하나 | `Carvedilol` |
| `Disease` | 질병 | 병 하나 | `hypertension`(고혈압) |
| `Gene` | 유전자 | 유전자 하나. 그 유전자가 만드는 **단백질까지 이 이름으로** 부릅니다 | `ADRB1` |
| `Symptom` | 증상 | 환자에게 나타나는 증상 | `Edema`(부종) |
| `PharmacologicClass` | 약효분류 | 작용 방식이 같은 약을 묶은 것 | `Calcium Channel Antagonists`(칼슘채널 차단제) |

열두 종류의 관계는 각각 이런 뜻입니다.

| 관계 | 잇는 것 | 설명 |
|---|---|---|
| `INCLUDES` | 약효분류 → 약물 | 그 약이 이 약효분류에 **속한다**(작용 방식이 같은 약 묶음) |
| `TREATS` | 약물 → 질병 | 그 약을 그 병의 **치료제로 쓴다**(병 자체를 겨냥한다) |
| `PALLIATES` | 약물 → 질병 | 그 약을 그 병의 **증상 완화에만 쓴다**(병 자체는 그대로다) |
| `BINDS` | 약물 → 유전자 | 그 약이 그 유전자가 만드는 **단백질에 결합한다**(그 약의 표적) |
| `UPREGULATES_CG` | 약물 → 유전자 | 그 약을 쓰면 그 유전자의 **발현량이 늘어난다** |
| `DOWNREGULATES_CG` | 약물 → 유전자 | 그 약을 쓰면 그 유전자의 **발현량이 줄어든다** |
| `RESEMBLES_CC` | 약물 → 약물 | 두 약의 **화학 구조가 비슷하다** |
| `PRESENTS` | 질병 → 증상 | 그 병의 환자에게 그 **증상이 나타난다** |
| `ASSOCIATES` | 질병 → 유전자 | 그 유전자가 그 병과 **관련 있다고 유전 연구에서 보고됐다** |
| `UPREGULATES_DG` | 질병 → 유전자 | 그 병의 환자에게서 그 유전자의 **발현량이 정상보다 많다** |
| `DOWNREGULATES_DG` | 질병 → 유전자 | 그 병의 환자에게서 그 유전자의 **발현량이 정상보다 적다** |
| `RESEMBLES_DD` | 질병 → 질병 | 두 병이 **닮은 병으로 묶여 있다**(같은 기관이거나 함께 나타나는 일이 잦다) |

> 이름 끝의 `_CG`·`_DG` 는 **무엇과 무엇을 잇는지**를 적어 둔 것입니다. `C` 는 약물(Compound), `D` 는 질병(Disease), `G` 는 유전자(Gene) 입니다. 발현량이 달라진다는 같은 말이라도 **그렇게 만든 것이 약이냐, 그런 상태로 관찰된 것이 병이냐**가 다르므로 이름을 갈라 두었습니다.

**데이터 출처**

- 원본: **Hetionet v1.0** (Himmelstein et al., 2017) · https://het.io · 라이선스 **CC0 1.0**
- 2016년까지 쌓인 생물의학 지식을 한 그래프로 모은 것입니다.
- 여기서 쓰는 것은 원본(노드 47,031 · 관계 2,250,197)에서 관계 24종 가운데 **12종만** 남긴 조각(노드 15,540 · 관계 91,966)입니다.

> **부작용 관계**(어떤 약이 어떤 부작용을 일으키는가)는 **라이선스 때문에** 뺐습니다. 출처인 SIDER 4 가 `CC BY-NC-SA` 라 상업적 이용을 막습니다.

아래 세 준비 셀을 위에서부터 실행하세요. **연결 → 초기화 → 데이터 적재** 순서입니다. 반드시 **실습 전용 DB**에 연결하세요(초기화 셀이 그래프를 지웁니다). 적재 셀은 3초쯤 걸립니다.

In [ ]:
# [제공 코드] Neo4j 연결: 실행만 하세요. .env 로 연결하고 run_cypher 헬퍼를 만듭니다.
# 반드시 "실습 전용" DB 에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j.exceptions import ConstraintError  # UNIQUE 제약 위반 에러

# 1) 접속 정보: .env 를 환경변수로 올린다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# .env 를 못 읽어도 에러가 아니라 기본값으로 넘어간다. 마지막 줄의 주소를 눈으로 확인할 것
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북이 끝날 때까지 재사용할 통로 하나
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 키는 RETURN 의 별칭이다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요!
# 노드·관계에 더해 이 노트북이 만든 인덱스·제약까지 지웁니다(DB 기본 LOOKUP 인덱스는 그대로).
# 1) 제약 먼저. 제약이 남아 있으면 그 제약이 만든 인덱스를 따로 못 지운다
for _row in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _row["name"] + " IF EXISTS")
# 2) 남은 인덱스. LOOKUP 은 DB 기본이라 뺀다
for _row in run_cypher("SHOW INDEXES YIELD name, type WHERE type <> 'LOOKUP' RETURN name"):
    run_cypher("DROP INDEX " + _row["name"] + " IF EXISTS")
# 3) 노드·관계. 관계가 9만 개라 한 번에 담지 않고 2만 개씩 끊어 지운다
while True:
    # DETACH DELETE 는 매달린 관계까지 함께 지운다
    _left = run_cypher("MATCH (n) WITH n LIMIT 20000 DETACH DELETE n RETURN count(n) AS n")[0]["n"]
    if _left == 0:
        break
print("초기화 완료: 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 의료 지식그래프 적재: 실행만 하세요(3초쯤 걸립니다).
# data/ 의 CSV 두 개로 노드 15,540개·관계 91,966개를 만듭니다.
# 맨 앞 CREATE CONSTRAINT 와 MATCH 의 레이블이 왜 필요한지는 다음 시간에 실측으로 확인합니다.
from pathlib import Path

import pandas as pd

# 교안 폴더에서 실행하면 data/, 정답 폴더에서 실행하면 ../data 를 본다
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")

# 노드 종류(레이블) 5가지. 관계 타입 12가지는 양 끝의 레이블이 하나로 정해져 있다.
NODE_LABELS = ["Compound", "Disease", "Gene", "Symptom", "PharmacologicClass"]
REL_SPEC = {
    "TREATS": ("Compound", "Disease"),              # 약물이 질병을 치료한다(질병 자체를 조절)
    "PALLIATES": ("Compound", "Disease"),           # 약물이 증상을 완화한다(치료와는 다른 말)
    "BINDS": ("Compound", "Gene"),                  # 약물이 그 유전자 산물에 결합한다(표적)
    "UPREGULATES_CG": ("Compound", "Gene"),         # 약물이 유전자 발현을 올린다
    "DOWNREGULATES_CG": ("Compound", "Gene"),       # 약물이 유전자 발현을 내린다
    "RESEMBLES_CC": ("Compound", "Compound"),       # 두 약물의 화학 구조가 닮았다
    "ASSOCIATES": ("Disease", "Gene"),              # 질병과 유전자가 연관돼 있다고 보고됐다
    "UPREGULATES_DG": ("Disease", "Gene"),
    "DOWNREGULATES_DG": ("Disease", "Gene"),
    "RESEMBLES_DD": ("Disease", "Disease"),
    "PRESENTS": ("Disease", "Symptom"),             # 질병이 그 증상으로 나타난다
    "INCLUDES": ("PharmacologicClass", "Compound"),  # 약효분류가 그 약물을 포함한다
}

# 1) id 를 유일하게 만드는 제약을 레이블마다 먼저 건다(다음 시간에 자세히 배웁니다).
for _label in NODE_LABELS:
    run_cypher(f"CREATE CONSTRAINT {_label.lower()}_id IF NOT EXISTS "
               f"FOR (n:{_label}) REQUIRE n.id IS UNIQUE")
run_cypher("CALL db.awaitIndexes()")   # 제약이 다 준비될 때까지 기다린다

# 2) 노드: 레이블별로 모아 UNWIND 로 한 번에 보낸다.
node_rows = {label: [] for label in NODE_LABELS}   # 레이블 -> 그 레이블 노드 줄 목록
nodes = pd.read_csv(DATA_DIR / "hetionet_nodes.csv")   # 열은 id·name·label 셋
for row in nodes.to_dict("records"):
    node_rows[row["label"]].append({"id": row["id"], "name": row["name"]})
for label, rows in node_rows.items():
    for start in range(0, len(rows), 5000):     # 5,000줄씩 끊어 보낸다(한 번에 다 보내면 메모리를 크게 잡는다)
        # MERGE 는 없으면 만들고 있으면 그대로 둔다. 그래서 이 셀을 두 번 돌려도 노드가 늘지 않는다
        run_cypher(f"UNWIND $rows AS row MERGE (n:{label} {{id: row.id}}) SET n.name = row.name",
                   rows=rows[start:start + 5000])

# 3) 관계: 타입마다 양 끝 레이블을 REL_SPEC 에서 꺼내 MATCH 에 그대로 적는다.
edge_rows = {rel: [] for rel in REL_SPEC}   # 관계 타입 -> 그 타입의 (출발 id, 도착 id) 목록
edges = pd.read_csv(DATA_DIR / "hetionet_edges.csv")   # 열은 source·rel·target 셋
for row in edges.to_dict("records"):
    edge_rows[row["rel"]].append({"s": row["source"], "t": row["target"]})
for rel, rows in edge_rows.items():
    source_label, target_label = REL_SPEC[rel]   # 이 레이블을 MATCH 에 적어야 노드를 곧장 짚는다
    for start in range(0, len(rows), 5000):
        run_cypher(f"UNWIND $rows AS row "
                   f"MATCH (a:{source_label} {{id: row.s}}), (b:{target_label} {{id: row.t}}) "
                   f"MERGE (a)-[:{rel}]->(b)", rows=rows[start:start + 5000])

print("적재 완료: 노드", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"],
      "· 관계", run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"])

## 그래프 살펴보기

집계를 시작하기 전에 **무엇이 들어 있는지** 먼저 봅니다. 표를 읽을 때 `head()`·`info()` 로 훑는 것과 같은 자리입니다. 그래프에서는 **레이블별 건수·관계 타입별 건수·노드 생김새** 셋을 봅니다.

In [ ]:
# [제공 코드] 그래프에 무엇이 들어 있는지 훑어봅니다(실행만 하세요).
# 1) 레이블마다 몇 개인지. 노드마다 레이블이 하나뿐이라 labels(n)[0] 로 충분하다
for row in run_cypher("MATCH (n) RETURN labels(n)[0] AS 레이블, count(*) AS 노드수 ORDER BY 노드수 DESC"):
    print(row)

In [ ]:
# [제공 코드] (이어서)
# 2) 관계 타입마다 몇 개인지. type(x) 가 관계 종류 이름이다
for row in run_cypher("MATCH ()-[x]->() RETURN type(x) AS 관계, count(x) AS 관계수 ORDER BY 관계수 DESC"):
    print(row)

In [ ]:
# [제공 코드] (이어서)
# 3) 노드가 실제로 어떻게 생겼는지 종류마다 두 개씩. id 는 원본 데이터베이스의 식별자를 그대로 쓴다
# collect(...)[0..2] 는 모은 리스트에서 앞 두 개만 잘라 본다는 뜻이다(0 이상 2 미만)
# collect 는 3절에서 배웁니다. 여기서는 '이렇게 훑는다' 정도로만 보고 넘어가세요
for row in run_cypher("""
    MATCH (n) WITH labels(n)[0] AS 레이블, n ORDER BY n.name
    RETURN 레이블, collect(n.name)[0..2] AS 이름예시, collect(n.id)[0..2] AS id예시
    ORDER BY 레이블
"""):
    print(row)

> 노드 `id` 가 `Compound::DB00682` 처럼 생긴 것에 주목하세요. 앞은 노드 종류, 뒤는 **원래 데이터베이스의 식별자**입니다(약물은 DrugBank, 질병은 Disease Ontology, 유전자는 Entrez). 이름은 사람이 읽는 표시일 뿐이고, 이 그래프에서 하나를 콕 집는 키는 `id` 입니다. 왜 그런지는 다음 시간에 확인합니다.

---
# 1. 개수 세기: count

세는 일은 세 걸음으로 나눠 익힙니다. 먼저 **무엇을 기준으로 묶어 셀지**를 정하고(1-1), 짝이 없는 행이 섞일 때 **`count(*)` 와 `count(x)` 가 갈리는 것**을 보고(1-2), 마지막으로 그래프에서 특히 자주 틀리는 **`DISTINCT`** 를 봅니다(1-3).

## 1-1. 무엇을 묶어 셀 것인가: 그룹핑 키

### 왜 필요할까요?
"이 병에 쓸 수 있는 약이 몇 개인가", "이 약이 건드리는 유전자가 몇 개인가" 같은 물음은 **행의 개수**를 묻습니다. 9만 개가 넘는 관계를 다 꺼내 눈으로 셀 수는 없습니다. `count` 가 **한 번에** 세어 줍니다.

### 문법: count 세 가지 얼굴
| 표현 | 세는 대상 |
|---|---|
| `count(*)` | 패턴에 맞은 **행의 수** |
| `count(x)` | `x` 가 **null 이 아닌** 행의 수 |
| `count(DISTINCT x)` | `x` 의 **서로 다른 값**의 수 |

**항목별로 세려면** RETURN 에 **세는 기준**(비집계 항)을 함께 둡니다. 그 비집계 항이 곧 **그룹핑 키**가 되어, 같은 값끼리 묶어 셉니다. SQL 의 `GROUP BY` 를 따로 쓰지 않아도 됩니다.

<img src="images/그룹핑키_비집계항.png" width="760">

집계할 항목과 묶을 항목을 따로 지정하지 않는다는 것이 핵심입니다. **집계하지 않고 남겨 둔 항목**이 자동으로 묶는 기준이 됩니다.

In [ ]:
# 하나만 세기: 'hypertension' 을 치료한다고 보고된 약이 몇 개인가
# 세는 대상 c 는 약물 노드다. 비집계 항(묶는 기준)을 하나도 두지 않았으니 전체가 한 덩어리로 세어진다
print(run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(:Disease {name: 'hypertension'})
    RETURN count(c) AS 약물수
"""))

In [ ]:
# 항목별로 세기: 질병마다 치료 약물이 몇 개인지. d.name 이 비집계 항이라 그룹핑 키가 된다
rows = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    RETURN d.name AS 질병, count(c) AS 약물수
    ORDER BY 약물수 DESC, 질병 LIMIT 5
""")   # 동점은 이름순으로 안정시킨다
for row in rows:
    print(row)

> 정렬에 **보조 키**(`질병`)를 함께 둔 것에 주목하세요. 개수가 같아 순위가 흔들릴 때 이름순으로 **안정**시키기 위해서입니다. 보조 키가 없으면 같은 점수인 줄들의 순서가 실행마다 달라질 수 있습니다.

### 🖐️ 함께 따라하기: 증상마다 몇 개 질병에서 나타나나

앞의 데모는 **약물-질병** 축이었습니다. 따라하기는 **질병-증상** 축으로 같은 기술을 연습합니다.

`(d:Disease)-[:PRESENTS]->(s:Symptom)` 패턴에서 `s.name` 을 그룹핑 키로 두고 질병 수를 세어, **많은 순**(동점이면 이름순)으로 상위 5개를 출력하세요. 별칭은 `증상`·`질병수` 로 합니다.

확인 기준: 1위는 `Edema`(49)입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 로 (질병)-[:PRESENTS]->(증상) 패턴을 잡는다
# 2) RETURN 에 증상 이름(비집계 항)과 count(질병) 을 함께 둔다 -> 증상이 그룹핑 키가 된다
# 3) ORDER BY 질병수 DESC, 증상  으로 정렬하고 LIMIT 5
# 4) 결과를 한 줄씩 print

### ✅ 바로 확인 퀴즈

**1)** `RETURN d.name, count(c)` 에서 `d.name` 을 빼면 결과가 어떻게 달라지나요?

<details><summary>정답 보기</summary>

묶는 기준이 사라져 **전체가 한 덩어리**가 됩니다. 질병별 개수가 아니라 전체 합계 한 줄이 나옵니다. 비집계 항이 곧 그룹핑 키이기 때문입니다.

</details>

**2)** 질병별 개수를 내면서 **약효분류까지** 함께 묶어 보고 싶다면 어떻게 하나요?

<details><summary>정답 보기</summary>

`RETURN` 에 비집계 항을 **하나 더** 둡니다(`RETURN p.name, d.name, count(c)`). 그룹핑 키는 몇 개든 둘 수 있고, **집계하지 않고 남겨 둔 항 전부**가 묶는 기준이 됩니다.

</details>

---
## 1-2. `count(*)` 와 `count(x)` 는 다른 답을 낸다

### 왜 필요할까요?
짝이 없는 행이 섞이면 두 표현이 갈립니다. 그리고 **그 차이 자체가 답인 경우**가 많습니다. 지난 시간에 배운 `OPTIONAL MATCH` 와 함께 쓸 때 특히 그렇습니다.

<img src="images/count_세얼굴.png" width="760">

같은 자리를 놓고도 **무엇을 세느냐**에 따라 답이 셋으로 갈립니다. 세기 전에 "행인가, 값이 있는 것인가, 서로 다른 값인가"를 먼저 정하세요.

In [ ]:
# OPTIONAL MATCH 라 치료약이 하나도 없는 질병도 행으로 남고, 그 행의 c 는 null 이다
# count(*) 는 그 null 행까지 세고, count(c) 는 c 가 채워진 행만 센다
print(run_cypher("""
    MATCH (d:Disease)
    OPTIONAL MATCH (c:Compound)-[:TREATS]->(d)
    RETURN count(*) AS 행수, count(c) AS 약물수
"""))

In [ ]:
# 두 수의 차이(814 - 755 = 59)가 곧 '치료약이 하나도 없는 질병' 수다
# NOT (패턴) 은 그 패턴이 아예 하나도 없는 노드만 남긴다
print(run_cypher("""
    // 화살표가 d 를 향하니 '약이 이 병을 치료한다' 는 뜻이다
    MATCH (d:Disease) WHERE NOT (d)<-[:TREATS]-(:Compound)
    RETURN count(d) AS 치료약없는질병
"""))

질병 136개 중 **59개는 치료약이 하나도 연결돼 있지 않습니다**. `count(*)` 로 셌다면 그 59개도 "1건"으로 세어져 814가 나왔을 것입니다. **무엇을 세는지가 곧 질문의 뜻**입니다.

### ✅ 바로 확인 퀴즈

**1)** "치료약이 하나도 없는 질병까지 포함해 질병 수를 세라"는 요청에는 `MATCH` 와 `OPTIONAL MATCH` 중 무엇을 써야 하나요?

<details><summary>정답 보기</summary>

`OPTIONAL MATCH`. 그냥 `MATCH` 로 이으면 짝이 없는 질병은 행 자체가 사라져 세어지지 않습니다.

</details>

**2)** 위 조회에서 `count(*)` 는 814, `count(c)` 는 755 였습니다. 그 차이는 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

**짝이 하나도 없던 행의 수**입니다(814 - 755 = 59). `OPTIONAL MATCH` 가 남겨 둔 그 행들의 `c` 는 `null` 이라 `count(c)` 가 세지 않습니다. 두 수를 나란히 내면 "없는 것이 몇 개인가"를 따로 세지 않고도 알 수 있습니다.

</details>

---
## 1-3. 노드 개수를 셀 때는 `DISTINCT`

### 왜 필요할까요?
그래프는 경로가 갈라집니다. 갈라진 만큼 같은 노드가 여러 행에 실려 오고, 그대로 세면 **부풀려집니다.** 표에서는 잘 안 나는 실수인데 그래프에서는 아주 흔합니다.

In [ ]:
# count(DISTINCT x): 경로가 갈라지면 같은 노드가 여러 번 세어진다
# 세는 대상 g 는 유전자 노드다: '몇 번 지나갔나' 와 '몇 개인가' 를 나란히 낸다
print(run_cypher("""
    // 천식을 치료하는 약(화살표가 천식을 향한다)을 거쳐 그 약이 결합하는 유전자까지 두 홉을 간다
    MATCH (:Disease {name: 'asthma'})<-[:TREATS]-(:Compound)-[:BINDS]->(g:Gene)
    RETURN count(g) AS 세어진횟수, count(DISTINCT g) AS 서로다른유전자
"""))

### ✅ 바로 확인 퀴즈

**1)** 한 질병에 `TREATS` 관계가 3개, `PALLIATES` 관계가 5개 있습니다. `MATCH (c:Compound)-[:TREATS|PALLIATES]->(d)` 로 잡고 `count(c)` 를 세면 몇이 나올까요?

<details><summary>정답 보기</summary>

8. 두 관계를 한 패턴에 이으면 행이 합쳐져 셉니다. **약이 몇 개인지**를 묻고 싶다면 `count(DISTINCT c)` 여야 하고, 한 약이 두 관계를 다 가지면 그때 답이 달라집니다.

</details>

**2)** 한 홉짜리 패턴(`(c)-[:TREATS]->(d)`)에서도 `DISTINCT` 를 꼭 붙여야 하나요?

<details><summary>정답 보기</summary>

그 패턴에서는 같은 노드가 두 번 걸릴 수 없어 **값이 같습니다.** 다만 패턴이 길어지는 순간 달라지므로, "경로가 갈라지는가"를 먼저 보고 판단하는 습관이 안전합니다.

</details>

---
# 2. 수치 요약: min·max·avg·sum

수치 요약은 세 걸음입니다. 1절에서 만든 수를 `WITH` 로 넘겨 **다시 집계하고**(2-1), 그 집계가 **조용히 틀리는 두 자리**(빠진 값과 정수 나눗셈)를 확인하고(2-2), 평균 하나가 가린 것을 **중앙값과 분위수**로 드러냅니다(2-3).

## 2-1. 집계한 수를 다시 집계하기

### 왜 필요할까요?
1절에서 질병마다 치료 약물 수를 냈습니다. 그런데 질병이 136개나 되니 목록을 다 보고 있을 수는 없습니다. "제일 많은 건 몇 개고, 평균은 몇 개인가"처럼 **그 수들을 다시 한 줄로 요약**하고 싶습니다.

이 그래프에는 가격이나 평점 같은 **숫자 속성이 없습니다.** 그래서 숫자는 우리가 만들어야 합니다. 1절의 `count` 로 만든 수를 `WITH` 로 넘겨 **한 번 더 집계**하는 것이 이 절의 전부입니다.

### 문법: 네 가지 수치 집계
| 함수 | 뜻 |
|---|---|
| `min(x)` | 가장 작은 값 |
| `max(x)` | 가장 큰 값 |
| `avg(x)` | 평균 |
| `sum(x)` | 합 |

핵심은 **순서**입니다. `MATCH` 로 행을 잡고 -> `WITH` 로 1차 집계해 수를 만들고 -> `RETURN` 에서 그 수를 다시 집계합니다. `WITH` 가 없으면 1차 집계 결과를 가리킬 이름이 없습니다.

In [ ]:
# 1단계만: 질병별 치료 약물 수 (1-1 에서 본 그것)
# 다음 셀에서 이 '약물수' 들을 다시 집계한다. 여기서는 그 재료가 어떻게 생겼는지만 본다
rows = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    RETURN d.name AS 질병, count(c) AS 약물수
    ORDER BY 약물수 DESC, 질병 LIMIT 3
""")
for row in rows:
    print(row)

In [ ]:
# WITH 뒤의 d 와 약물수 가 다음 단계로 넘어가는 행이 되고, RETURN 이 그 행들을 한 줄로 요약한다
# round(avg(...), 2) 로 소수 둘째 자리까지만 남긴다. avg 를 그냥 두면 자릿수가 길다
print(run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WITH d, count(c) AS 약물수
    RETURN count(d) AS 질병수, min(약물수) AS 약물수최소, max(약물수) AS 약물수최대,
           round(avg(약물수), 2) AS 약물수평균, sum(약물수) AS 약물수합계
"""))

- **질병수 77**: 치료약이 하나라도 있는 질병만 세어졌습니다(1-2 의 59개는 패턴에 안 맞아 빠집니다).
- **약물수최소 1 / 약물수최대 68**: 약이 하나뿐인 병도 있고 `hypertension` 처럼 68개인 병도 있습니다.
- **약물수평균 9.81**: `avg` 는 소수가 나오므로 `round(값, 자릿수)` 로 다듬습니다.
- **약물수합계 755**: 이 값은 `TREATS` 관계의 전체 개수와 같아야 합니다. **검산이 되는 자리**입니다.

집계 결과가 맞는지 의심스러울 때는 이렇게 **다른 방법으로 같은 수가 나오는지** 확인하는 습관이 좋습니다.

In [ ]:
# 검산: 위의 '약물수합계' 가 TREATS 관계 전체 개수와 같은가
# 질병마다 센 값을 모두 더한 것이라, 행이 새거나 겹치지 않았다면 관계 수와 정확히 같아야 한다
print(run_cypher("MATCH (:Compound)-[r:TREATS]->(:Disease) RETURN count(r) AS 전체TREATS"))

### 🖐️ 함께 따라하기: 증상은 평균 몇 개 질병에서 나타나나

질병-증상 축으로 같은 2단계 집계를 해 보세요.

`(d:Disease)-[:PRESENTS]->(s:Symptom)` 패턴에서 `WITH s, count(d) AS 질병수` 로 1차 집계한 뒤, `RETURN` 에서 증상 개수와 그 질병 수의 최소·최대·평균(소수 둘째 자리)·합계를 냅니다. 별칭은 `증상수`·`질병수최소`·`질병수최대`·`질병수평균`·`질병수합계` 로 합니다.

확인 기준: 증상수 `415`, 질병수최대 `49`, 질병수합계 `3357`(= `PRESENTS` 관계 수)입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 로 (질병)-[:PRESENTS]->(증상) 패턴을 잡는다
# 2) WITH s, count(d) AS 질병수  로 증상마다 몇 개 질병에서 나타나는지 먼저 센다
# 3) RETURN 에서 count(s)·min·max·round(avg(...), 2)·sum 을 한 줄로 낸다
# 4) 합계가 PRESENTS 관계 전체 수와 같은지 따로 조회해 검산한다

### ✅ 바로 확인 퀴즈

**1)** `MATCH (c)-[:TREATS]->(d) RETURN avg(count(c))` 는 왜 안 될까요?

<details><summary>정답 보기</summary>

집계 함수를 **집계 함수 안에 겹쳐 쓸 수 없기** 때문입니다. 먼저 `WITH d, count(c) AS 약물수` 로 1차 집계를 끝내 이름을 붙인 뒤, 다음 단계에서 `avg(약물수)` 를 씁니다.

</details>

**2)** 위 요약에서 `합계` 가 `755` 였습니다. 이 수는 무엇과 같아야 하나요?

<details><summary>정답 보기</summary>

`TREATS` 관계의 전체 개수. 질병마다 센 값을 모두 더한 것이므로, 중간에 행이 새거나 겹치지 않았다면 관계 수와 정확히 같습니다. 검산에 쓸 수 있습니다.

</details>

**3)** 평균이 `9.81` 인데 최대는 `68` 입니다. 이 평균을 "보통 질병에는 약이 9개쯤 있다"고 읽어도 될까요?

<details><summary>정답 보기</summary>

조심해야 합니다. 최댓값이 평균의 몇 배나 되면 **소수의 큰 값이 평균을 끌어올린** 것입니다. 실제로 치료약이 하나뿐인 질병이 최솟값 1로 존재합니다. 이럴 때는 평균 하나만 보지 말고 분포를 함께 봐야 합니다. 바로 다음 2-3 에서 그 분포를 한 줄로 봅니다.

</details>

---
## 2-2. 집계가 조용히 틀리는 두 자리

### 왜 필요할까요?
집계에는 **에러 없이 답만 틀리는** 자리가 둘 있습니다. 통째로 실행해도 아무 표시가 나지 않아, 알고 있지 않으면 찾을 방법이 없습니다.

**첫째, 평균은 분모가 조용히 바뀝니다.** 같은 물음이라도 **어떤 행을 남기느냐**에 따라 나누는 수가 달라집니다. 방금 낸 평균으로 바로 확인할 수 있습니다.

<img src="images/같은_합계_다른_평균.png" width="760">

같은 물음인데 답이 갈립니다. 더한 값(분자)은 똑같고, **몇 개로 나눴는지(분모)만** 다릅니다. 평균을 볼 때는 늘 "몇 개를 평균 낸 것인가"를 함께 보세요.

In [ ]:
# 1) MATCH: 치료약이 있는 질병만 행이 된다 -> 분모가 그만큼 작다
strict = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WITH d, count(c) AS 약물수
    RETURN count(d) AS 질병수, round(avg(약물수), 2) AS 약물수평균,
           sum(약물수) AS 약물수합계
""")
# 2) OPTIONAL MATCH: 치료약이 없는 질병도 행으로 남아 0 으로 세어진다 -> 분모가 전체다
loose = run_cypher("""
    MATCH (d:Disease)
    OPTIONAL MATCH (c:Compound)-[:TREATS]->(d)
    WITH d, count(c) AS 약물수
    RETURN count(d) AS 질병수, round(avg(약물수), 2) AS 약물수평균,
           sum(약물수) AS 약물수합계
""")
print("치료약이 있는 질병만:", strict[0])
print("질병 전체:        ", loose[0])

**합계는 755 로 같은데 평균은 9.81 과 5.55 로 갈립니다.** 분자는 같고 분모만 77 대 136 로 다르기 때문입니다.

둘 다 맞는 답입니다. 다만 **다른 질문에 답하고 있습니다.** "치료약이 있는 질병은 평균 몇 개를 가졌나"와 "질병 하나당 평균 몇 개가 붙어 있나"는 다른 물음입니다. 평균을 낼 때는 **분모가 무엇인지**를 늘 함께 확인하세요. 그래서 `count(*)` 를 평균 옆에 나란히 두는 습관이 좋습니다.

빠진 값을 만났을 때 집계 함수가 어떻게 하는지는 이렇습니다.

| 표현 | 빠진 값을 만나면 |
|---|---|
| `avg(x)` | 건너뛴다. 행 수가 아니라 **값이 있는 것의 개수**로 나눈다 |
| `sum(x)` | 건너뛴다. **다 빠져 있으면 `null` 이 아니라 `0`** 이 나온다 |
| `min(x)`·`max(x)` | 건너뛴다 |
| `collect(x)` | 건너뛴다. `null` 은 목록에 담기지 않는다 |
| `count(x)` | 세지 않는다 |
| `count(*)` | **센다**(행이니까) |

> `sum` 이 `0` 을 내는 것이 특히 위험합니다. "합이 0원"과 "금액이 아예 안 들어왔다"는 전혀 다른 상황인데 결과만 보면 구분되지 않습니다. **`count(x)` 를 옆에 두어 몇 개를 더한 것인지** 함께 보세요.

### 두 번째 자리: 정수끼리 나누면 소수점이 버려진다

방금 낸 평균 `9.81` 을 직접 계산해 보겠습니다. 합계 755 를 질병수 77 로 나누면 됩니다.

In [ ]:
# 합계와 질병수를 손으로 나눠 본다. 둘 다 정수다
# Cypher 는 정수끼리 나누면 소수점 아래를 버린다(파이썬의 // 와 같다). 에러는 나지 않는다
print(run_cypher("""
    RETURN $합계 / $질병수 AS 정수나눗셈,
           round(toFloat($합계) / $질병수, 2) AS 실수나눗셈
""", 합계=755, 질병수=77))


`755 / 77` 가 **`9`** 입니다. `avg` 가 낸 `9.81` 과 다릅니다. Cypher 는 **정수끼리 나누면 소수점 아래를 버립니다.**

`avg`·`round` 를 쓰는 동안에는 이 함정을 만나지 않습니다. 문제는 **비율을 직접 만들 때**입니다. `공유수 / 전체수` 처럼 쓰면 거의 언제나 `0` 이 나오고, 모든 항목이 동점이 되어 순위가 뜻을 잃습니다. 그런데 **에러는 나지 않습니다.**

고치는 법은 한 가지입니다. **어느 한쪽을 `toFloat` 로 감싸세요**(`toFloat(a) / b`). 다음 교안의 랭킹 절에서 이 모양을 그대로 씁니다.

### 건너뛰지 말고 채우기: `coalesce`

앞의 빠진 값 표는 전부 "건너뛴다" 였습니다. 그런데 건너뛰는 것이 늘 옳지는 않습니다. "값이 없다"를 **0 으로 봐야 하는** 물음도 있고, 표에 찍을 때 빈칸 대신 **글자를 채워야** 할 때도 있습니다. 그럴 때는 집계에 넣기 전에 빈자리를 메웁니다.

| 표현 | 뜻 |
|---|---|
| `coalesce(a, b)` | `a` 가 `null` 이 아니면 `a`, 그렇지 않으면 `b` |
| `coalesce(a, b, c)` | 앞에서부터 처음으로 `null` 이 아닌 값 |

**집계 함수 안이 아니라 행마다 먼저** 걸린다는 점이 중요합니다. `avg(coalesce(x, 0))` 은 "빈자리를 0 으로 바꾼 다음 평균"을 냅니다.

In [ ]:
# 1-2 에서 쓴 그 조회 그대로다. 달라진 것은 coalesce 를 씌운 칸 하나뿐이다
# 치료약이 없는 질병은 c 가 null 이라 c.name 도 null 이고, count 가 그 자리를 세지 않는다
# coalesce 는 null 을 내지 않으므로 채운 칸은 결국 count(*) 와 같은 수가 된다
print(run_cypher("""
    MATCH (d:Disease)
    OPTIONAL MATCH (c:Compound)-[:TREATS]->(d)
    RETURN count(*) AS 행수, count(c.name) AS 채우기전,
           count(coalesce(c.name, '(치료약 없음)')) AS 채운뒤
"""))

채우기 전은 755, 채운 뒤는 814 입니다. **1-2 에서 `count(c)` 와 `count(*)` 가 갈렸던 그 두 수**입니다. 빈자리를 채우자 작은 쪽이 큰 쪽으로 올라왔습니다.

규칙 하나로 외워 두면 편합니다. **`coalesce` 를 씌우면 `count(x)` 가 `count(*)` 와 같아집니다.** 채운 값은 `null` 이 아니므로 빠짐없이 세어지기 때문입니다.

무엇이 옳은지는 데이터가 정해 주지 않습니다. 빠진 값이 **"아직 모른다"면 건너뛰고, "0 이다"면 채웁니다.** 온도를 못 잰 날을 0도로 채우면 평균이 망가지고, 팔린 것이 없던 날을 건너뛰면 일평균 매출이 부풀려집니다.

채운 값이 표에 실제로 어떻게 찍히는지 봅니다.

In [ ]:
# 질병 하나를 한 줄로 보이려면 여러 약 이름을 값 하나로 접어야 한다. 그 접는 일을 min 이 한다
# min 이라서 이름순 첫 값이 뽑힐 뿐, 어느 것을 고르느냐는 여기서 중요하지 않다
# 접을 값이 아예 없으면 min 은 null 을 낸다. 그 자리를 coalesce 로 메운다
rows = run_cypher("""
    MATCH (d:Disease)
    OPTIONAL MATCH (c:Compound)-[:TREATS]->(d)
    WITH d, count(c) AS 약물수, min(c.name) AS 대표약
    RETURN d.name AS 질병, 약물수, coalesce(대표약, '(치료약 없음)') AS 대표약
    ORDER BY 약물수, 질병 LIMIT 5
""")
for row in rows:
    print(row)

`약물수` 에는 `coalesce` 를 쓰지 않았습니다. `count` 는 셀 것이 없으면 `null` 이 아니라 **`0`** 을 내기 때문입니다. 채워야 하는 것은 `min`·`max`·`avg` 처럼 **값이 없으면 `null` 을 내는** 집계와, **아예 붙어 있지 않은 속성**입니다. 어디에나 `coalesce` 를 둘러 두면 코드만 길어지고 무엇이 진짜 빈자리였는지 흐려집니다.

### 🖐️ 함께 따라하기: 증상 축에서 분모를 확인하기

질병-증상 축에서 같은 대비를 만들어 보세요.

1. `MATCH (d:Disease)-[:PRESENTS]->(s:Symptom)` 로 **증상이 하나라도 있는 질병만** 잡아 질병마다 증상 수를 세고, 그 수들의 `count`·`평균`(소수 둘째 자리)을 냅니다.
2. 같은 것을 `MATCH (d:Disease) OPTIONAL MATCH (d)-[:PRESENTS]->(s:Symptom)` 로 **질병 전체**에 대해 냅니다.
3. 두 결과를 나란히 출력하고, 합계가 같은지도 함께 확인합니다.

별칭은 `질병수`·`증상수평균`·`증상수합계` 로 합니다. 확인 기준: 두 합계는 같고 두 평균은 다릅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 로 증상이 있는 질병만 잡아 WITH d, count(s) AS 증상수 로 1차 집계한 뒤
#    RETURN count(d) AS 질병수, round(avg(증상수), 2) AS 증상수평균, sum(증상수) AS 증상수합계
# 2) 같은 것을 MATCH (d:Disease) + OPTIONAL MATCH 로 질병 전체에 대해 낸다
# 3) 두 결과를 나란히 print 한다(합계가 같은지 확인)

### ✅ 바로 확인 퀴즈

**1)** 관계 100건 중 90건에만 `price` 속성이 있습니다. `avg(r.price)` 는 무엇으로 나눈 값인가요?

<details><summary>정답 보기</summary>

**90** 으로 나눈 값입니다. 속성이 없는 10건은 `null` 이라 건너뜁니다. "100건의 평균"이라고 읽으면 틀립니다. `count(r.price)` 를 옆에 두면 몇 개를 평균 낸 것인지 바로 보입니다.

</details>

**2)** 어떤 관계에도 `price` 가 없다면 `sum(r.price)` 는 무엇을 돌려주나요?

<details><summary>정답 보기</summary>

`null` 이 아니라 **`0`** 입니다. "합이 0"과 "값이 하나도 없다"가 구분되지 않으므로, `count(r.price)` 를 함께 보아야 합니다.

</details>

**3)** `RETURN 공유수 / 전체수 AS 비율` 이 전부 `0` 으로 나옵니다. 무엇이 문제인가요?

<details><summary>정답 보기</summary>

정수끼리 나눠 소수점 아래가 버려졌습니다. 공유수가 전체수보다 작으니 몫이 전부 0 입니다. **`toFloat(공유수) / 전체수`** 로 한쪽을 실수로 만들어야 합니다. 에러가 나지 않는 것이 이 함정의 특징입니다.

</details>

---
## 2-3. 평균이 가린 것: 중앙값과 분위수

### 왜 필요할까요?
평균은 `9.81` 인데 최대는 `68` 입니다. **소수의 큰 값이 평균을 끌어올린 것**인지, 평균과 최대만으로는 알 수 없습니다.

이럴 때 보는 값이 **중앙값**입니다. 값들을 줄 세웠을 때 한가운데 오는 값이라 큰 값 몇 개에 끌려가지 않습니다.

### 문법: 분위수 집계
| 함수 | 뜻 |
|---|---|
| `percentileCont(x, 0.5)` | 중앙값(50% 지점). 값 사이는 직선으로 이어 구한다 |
| `percentileCont(x, 0.9)` | 90% 지점. 상위 10% 가 어디서부터인지 |
| `stDev(x)` | 표준편차. 값들이 평균에서 얼마나 흩어져 있나 |

`0.5`·`0.9` 자리에는 0 과 1 사이의 값을 넣습니다. 다른 집계 함수와 똑같이 `WITH` 로 만든 수에도 쓸 수 있습니다.

In [ ]:
# 2-1 과 같은 '질병별 치료 약물 수' 를 이번에는 분포까지 함께 본다
# percentileCont 의 둘째 인자는 0~1 사이의 지점이다. 0.5 가 중앙값, 0.9 가 상위 10% 경계
print(run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WITH d, count(c) AS 약물수
    RETURN round(avg(약물수), 2) AS 약물수평균,
           percentileCont(약물수, 0.5) AS 약물수중앙값,
           percentileCont(약물수, 0.9) AS 약물수p90,
           max(약물수) AS 약물수최대, round(stDev(약물수), 2) AS 약물수표준편차
"""))


이제 2-1 의 퀴즈에 숫자로 답할 수 있습니다. 평균은 `9.81` 이지만 **중앙값은 `7`** 입니다. 절반의 질병은 치료약이 `7`개 이하라는 뜻입니다. 평균이 중앙값보다 큰 것은 `hypertension`(68)처럼 유난히 큰 값 몇 개가 평균을 끌어올렸기 때문입니다.

`p90` 이 `21.0` 이라는 것도 읽어 둘 만합니다. 질병의 90% 는 치료약이 `21`개 이하이고, 상위 10% 만 그보다 많습니다.

> **평균 하나만 적힌 보고서는 믿지 마세요.** 최소·중앙값·최대를 함께 내는 데 드는 비용은 쿼리 한 줄이고, 그 한 줄이 "평균 9개" 라는 오해를 막습니다. 6-1 에서는 이 분포를 **구간 라벨**로 나눠 더 자세히 봅니다.

### ✅ 바로 확인 퀴즈

**1)** 평균 `9.81` 과 중앙값 `7` 중 "보통 질병" 을 말하는 데 더 알맞은 값은 무엇인가요?

<details><summary>정답 보기</summary>

**중앙값 `7`** 입니다. 평균은 큰 값 몇 개에 끌려 올라갑니다. 값이 한쪽으로 치우친 자료에서는 중앙값이 "가운데"를 더 잘 나타냅니다.

</details>

**2)** `percentileCont(약물수, 0)` 과 `percentileCont(약물수, 1)` 은 각각 무엇과 같나요?

<details><summary>정답 보기</summary>

`min(약물수)` 와 `max(약물수)` 입니다. 0% 지점이 가장 작은 값, 100% 지점이 가장 큰 값입니다.

</details>

---
# 3. 리스트로 모으고 다시 펼치기: collect·UNWIND

리스트를 다루는 일은 네 걸음입니다. 값들을 **리스트로 모으고**(3-1), 그 리스트를 **다시 행으로 펴고**(3-2), 리스트를 **걸러 쓰고**(3-3), 마지막으로 패턴을 **바로 리스트로 받는 법**(3-4)을 봅니다.

## 3-1. collect 로 모으기

### 왜 필요할까요?
`count` 는 "몇 개"만 알려 줍니다. "그래서 **무엇들**인가"를 묻고 싶을 때가 있습니다. `collect` 는 묶인 값들을 **리스트 한 칸**으로 모아 줍니다.

### 문법: collect 와 UNWIND
| 표현 | 하는 일 |
|---|---|
| `collect(x)` | 묶음 안의 `x` 들을 **리스트 하나**로 모은다 |
| `size(리스트)` | 리스트의 길이 |
| `리스트[0..3]` | 리스트 앞부분만 잘라 본다(0 이상 3 미만) |
| `UNWIND 리스트 AS x` | 리스트를 **한 줄씩 행으로** 펼친다(collect 의 반대) |

<img src="images/count_대_collect.png" width="760">

같은 묶음을 놓고 `count` 는 숫자 한 칸을, `collect` 는 리스트 한 칸을 돌려줍니다. "몇 개인지"를 물으면 `count`, "무엇들인지"를 물으면 `collect` 입니다.

In [ ]:
# 질병마다 치료 약물 '목록'을 리스트로 모은다. 목록이 길 수 있어 앞 5개만 잘라 본다
# count 는 '몇 개인지', collect 는 '무엇들인지'. 같은 묶음이라 한 RETURN 에 나란히 쓸 수 있다
rows = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    RETURN d.name AS 질병, count(c) AS 약물수, collect(c.name)[0..5] AS 약물맛보기
    ORDER BY 약물수 DESC, 질병 LIMIT 3
""")
for row in rows:
    print(row)

In [ ]:
# size 로 리스트 길이를 센다: 목록을 만들어 두고 그 길이를 함께 낼 때 쓴다
# WITH 에서 접어 둔 리스트를 RETURN 에서 재는 것이라, 값은 count(c) 로 센 것과 같다
rows = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WITH d, collect(c.name) AS 약물목록
    RETURN d.name AS 질병, size(약물목록) AS 약물수
    ORDER BY 약물수 DESC, 질병 LIMIT 3
""")
for row in rows:
    print(row)

리스트를 모을 때도 `DISTINCT` 를 쓸 수 있습니다. 1-3 에서 본 `count(DISTINCT x)` 와 같은 자리입니다.

In [ ]:
# collect(DISTINCT x): 같은 값이 여러 행에 실려 와도 한 번씩만 모은다
# 천식 치료약들이 결합하는 유전자는 여러 약이 겹쳐 붙어 같은 이름이 여러 번 실려 온다
print(run_cypher("""
    // 천식을 치료하는 약(화살표가 천식을 향한다)을 거쳐 그 약이 결합하는 유전자까지 두 홉을 간다
    MATCH (:Disease {name: 'asthma'})<-[:TREATS]-(:Compound)-[:BINDS]->(g:Gene)
    RETURN size(collect(g.name)) AS 그냥모으면,
           size(collect(DISTINCT g.name)) AS 중복빼고모으면
"""))


---
## 3-2. UNWIND 로 다시 행으로 펴기

### 왜 필요할까요?
`collect` 가 여러 행을 한 칸으로 접었다면, `UNWIND` 는 그 칸을 **다시 여러 행으로 폅니다.** 접었다 펴면 원래 행 수로 돌아옵니다.

In [ ]:
# 접었다 펴면 원래대로: collect 로 모은 리스트를 UNWIND 로 되펼친다
print(run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    RETURN count(*) AS 원래행수
"""))
print(run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WITH d, collect(c.name) AS 약물목록          // 접는다
    UNWIND 약물목록 AS 약물이름                  // 편다
    RETURN count(*) AS 펼친행수
"""))

두 수가 모두 `755` 로 같습니다. `collect` 와 `UNWIND` 는 서로를 되돌리는 짝입니다.

In [ ]:
# UNWIND 로 리스트를 행으로 펼친 뒤, 그 행마다 MATCH 를 돌린다 (파이썬 for 문을 Cypher 안에 넣은 셈)
# 펼쳐 놓은 병명 은 그냥 값이라 MATCH 안의 {name: 병명} 자리에 그대로 쓸 수 있다
rows = run_cypher("""
    UNWIND ['hypertension', 'asthma', 'migraine'] AS 병명
    MATCH (c:Compound)-[:TREATS]->(d:Disease {name: 병명})
    RETURN 병명, count(c) AS 치료약물수 ORDER BY 치료약물수 DESC
""")
for row in rows:
    print(row)

목록을 쿼리 안에 박아 두면 다른 병을 보려 할 때마다 쿼리 문자열을 고쳐야 합니다. 지난 단원에서 배운 **파라미터**로 넘기면 쿼리는 한 벌 그대로 두고 파이썬 쪽 값만 바꿉니다. 결과는 앞 셀과 같고, 달라지는 것은 목록이 어디에 있느냐뿐입니다.

In [ ]:
# 목록을 쿼리에 박지 않고 파이썬 변수에서 넘긴다. $병명목록 이 그 자리를 잡아 두는 표시다
# 쿼리 문자열은 한 글자도 바뀌지 않고, 고치는 것은 파이썬 리스트(disease_names)뿐이다
disease_names = ['hypertension', 'asthma', 'migraine']
rows = run_cypher("""
    UNWIND $병명목록 AS 병명
    MATCH (c:Compound)-[:TREATS]->(d:Disease {name: 병명})
    RETURN 병명, count(c) AS 치료약물수 ORDER BY 치료약물수 DESC
""", 병명목록=disease_names)
for row in rows:
    print(row)

> 이 모양은 **대량 적재**에서도 그대로 쓰입니다. 오늘 맨 위 적재 셀이 CSV 9만 줄을 `UNWIND $rows AS row` 로 한 번에 밀어 넣은 것이 바로 이것입니다. 파이썬에서 9만 번 왕복하는 것보다 훨씬 빠릅니다.

### 🖐️ 함께 따라하기: 증상마다 어떤 질병에서 나타나나

질병-증상 축으로 `collect` 를 연습합니다.

`(d:Disease)-[:PRESENTS]->(s:Symptom)` 에서 증상마다 `질병수`(count)와 `질병맛보기`(`collect(d.name)[0..4]`)를 함께 내어, 질병수 내림차순(동점은 이름순) 상위 3개를 출력하세요.

확인 기준: 1위는 `Edema`(49)입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH 로 (질병)-[:PRESENTS]->(증상) 패턴을 잡고
#    RETURN 에 증상 이름·count(질병)·collect(질병이름)[0..4] 를 함께 둔다
# 2) ORDER BY 질병수 DESC, 증상 LIMIT 3 으로 정렬해 print

### ✅ 바로 확인 퀴즈

**1)** `collect(c.name)` 과 `count(c)` 를 한 RETURN 에 같이 쓸 수 있나요?

<details><summary>정답 보기</summary>

쓸 수 있습니다. 둘 다 같은 묶음을 대상으로 하는 집계라 한 줄에 함께 나옵니다. "몇 개인지"와 "무엇들인지"를 한 번에 보고 싶을 때 자주 쓰는 조합입니다.

</details>

**2)** `UNWIND [] AS x MATCH ...` 처럼 **빈 리스트**를 펼치면 어떻게 되나요?

<details><summary>정답 보기</summary>

행이 하나도 생기지 않아 뒤 절이 아예 실행되지 않습니다. 결과는 빈 목록입니다. 파라미터로 받은 리스트가 비어 있을 수 있다면 이 점을 염두에 두세요.

</details>

**3)** 목록이 수십 개라 화면이 넘칠 때 앞 5개만 보려면 어떻게 쓰나요?

<details><summary>정답 보기</summary>

`collect(c.name)[0..5]`. 리스트 뒤에 `[시작..끝]` 을 붙이면 잘라 볼 수 있습니다(끝은 포함하지 않습니다).

</details>

---
## 3-3. 리스트를 걸러 쓰기: 리스트 컴프리헨션

### 왜 필요할까요?
모아 둔 리스트에서 **일부만** 쓰고 싶을 때가 있습니다. `[0..5]` 로 자르는 것은 앞에서 몇 개를 떼는 것뿐이고, "이름이 A 로 시작하는 것만" 같은 **조건**은 걸 수 없습니다.

파이썬의 리스트 컴프리헨션(`[x for x in 목록 if 조건]`)을 이미 배웠습니다. Cypher 에도 같은 것이 있고 모양도 닮았습니다.

### 문법: 세로줄로 나눈다
```
[x IN 목록 WHERE 조건 | 바꿀식]
```
| 조각 | 하는 일 | 파이썬으로 치면 |
|---|---|---|
| `x IN 목록` | 하나씩 꺼낸다 | `for x in 목록` |
| `WHERE 조건` | 조건에 맞는 것만 남긴다(생략 가능) | `if 조건` |
| `| 바꿀식` | 남은 것을 바꾼다(생략 가능) | 맨 앞의 식 |

그리고 리스트에서 한 칸만 꺼낼 때는 **자리 번호**를 씁니다. `목록[0]` 이 첫 번째, `목록[-1]` 이 마지막입니다.

<img src="images/컴프리헨션_두_가지.png" width="760">

대괄호가 하는 일이 둘입니다. **이미 있는 목록을 거르는 것**(3-3)과 **패턴에서 목록을 바로 만드는 것**(3-4). 모양이 닮아 헷갈리기 쉬우니 무엇이 대괄호 앞에 오는지로 가르세요.

In [ ]:
# 편두통 치료약 목록을 모아 여러 방법으로 다뤄 본다
# collect 는 담기는 순서를 보장하지 않는다. 첫/마지막을 꺼낼 것이니 접기 전에 정렬해 둔다
print(run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(:Disease {name: 'migraine'})
    WITH c.name AS 이름 ORDER BY 이름
    WITH collect(이름) AS 약물들
    RETURN size(약물들) AS 약물수,
           [x IN 약물들 WHERE x STARTS WITH 'A'] AS A로시작,
           약물들[0] AS 첫번째, 약물들[-1] AS 마지막
"""))


In [ ]:
# 세로줄 뒤에 식을 적으면 값을 바꿔서 모은다. 여기서는 이름 대신 이름의 길이를 담는다
# 자리 번호가 리스트 길이를 넘으면 에러가 아니라 null 이 나온다(파이썬과 다른 점이다)
print(run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(:Disease {name: 'migraine'})
    WITH c.name AS 이름 ORDER BY 이름
    WITH collect(이름) AS 약물들
    RETURN [x IN 약물들 | size(x)] AS 이름길이, 약물들[99] AS 범위밖
"""))


> **범위 밖 자리 번호가 `null` 이 되는 것**을 기억하세요. 파이썬은 `IndexError` 를 내지만 Cypher 는 조용히 `null` 을 줍니다. `목록[0]` 을 썼는데 결과가 `None` 이면 그 리스트가 비어 있었다는 뜻입니다.

### ✅ 바로 확인 퀴즈

**1)** `[x IN 약물들 WHERE x STARTS WITH 'A']` 에서 세로줄과 뒤의 식을 생략하면 무엇이 담기나요?

<details><summary>정답 보기</summary>

**걸러 낸 값 그대로** 담깁니다. 바꿀 식을 안 적으면 꺼낸 값을 그대로 씁니다(`| x` 를 적은 것과 같습니다).

</details>

**2)** `collect(c.name)[0..5]` 와 `[x IN collect(c.name) WHERE ...]` 는 무엇이 다른가요?

<details><summary>정답 보기</summary>

앞은 **자리로** 자르고(앞에서 5개), 뒤는 **조건으로** 거릅니다. "앞의 몇 개"를 원하면 앞쪽, "어떤 것들"을 원하면 뒤쪽입니다.

</details>

---
## 3-4. 패턴을 바로 리스트로: 패턴 컴프리헨션

### 왜 필요할까요?
"이 질병의 치료약 목록"을 내려면 지금까지는 `MATCH` 로 패턴을 잡고 `collect` 로 모아야 했습니다. 그런데 한 행에 **목록을 두 개 이상** 붙이고 싶으면 `MATCH` 를 여러 번 쓰게 되고, 짝이 없는 노드가 행째로 사라지지 않게 `OPTIONAL MATCH` 도 챙겨야 합니다.

**패턴 컴프리헨션**은 `RETURN` 자리에서 패턴을 바로 리스트로 받습니다. 모양은 리스트 컴프리헨션과 같고, `x IN 목록` 자리에 **패턴**이 들어갑니다.

### 문법
```
[(a)-[:관계]->(b) WHERE 조건 | 꺼낼식]
```
짝이 하나도 없으면 **에러가 아니라 빈 리스트**(`[]`)가 나옵니다. `OPTIONAL MATCH` 를 쓰던 자리를 대신할 수 있는 이유가 이것입니다.

In [ ]:
# 편두통 하나를 잡아, 그 병에 딸린 목록 셋을 한 줄에 받는다
# MATCH 는 한 번뿐이다. 나머지 세 목록은 RETURN 자리의 패턴이 각자 만들어 낸다
print(run_cypher("""
    MATCH (d:Disease {name: 'migraine'})
    // 세 패턴 모두 d 를 한쪽 끝에 두고, 화살표 방향으로 치료·증상·완화를 가른다
    RETURN [(c:Compound)-[:TREATS]->(d) | c.name] AS 치료약,
           [(d)-[:PRESENTS]->(s:Symptom) | s.name][0..3] AS 증상맛보기,
           size([(d)<-[:PALLIATES]-(c:Compound) | c.name]) AS 완화약수
"""))


In [ ]:
# 짝이 하나도 없는 노드에서는 어떻게 되나: 에러가 아니라 빈 리스트가 나온다
# NOT (패턴) 으로 '치료약이 하나도 없는 질병' 을 잡아 그 자리에서 확인한다
print(run_cypher("""
    // 화살표가 d 를 향하니 '약이 이 병을 치료한다' 는 뜻이다
    MATCH (d:Disease) WHERE NOT (d)<-[:TREATS]-(:Compound)
    RETURN d.name AS 질병, [(c:Compound)-[:TREATS]->(d) | c.name] AS 치료약
    ORDER BY 질병 LIMIT 1
"""))


여기서 말하는 **목록**은 결과 한 행에 딸려 나오는 **리스트 칸**입니다. 앞에서 낸 `치료약`·`증상맛보기` 같은 것입니다.

| 하고 싶은 일 | `MATCH` + `collect` | 패턴 컴프리헨션 |
|---|---|---|
| 한 행에 리스트 칸 **하나**(그 병의 치료약 이름들) | `collect(c.name)` 으로 모은다 | `[(c)-[:TREATS]->(d) \| c.name]` 로 바로 받는다 |
| 한 행에 리스트 칸 **여럿**(치료약·증상·완화약을 나란히) | `OPTIONAL MATCH` 를 여러 번 쓰고 `WITH` 로 단계를 나눈다 | 한 줄에 나란히 적는다 |
| 짝이 하나도 없는 노드도 행에 남기기 | `MATCH` 면 행째로 사라져 `OPTIONAL MATCH` 가 필요하다 | 빈 리스트 `[]` 가 된다 |
| 리스트 말고 **개수**만 필요할 때 | `count` 로 바로 센다 | `size(...)` 로 한 번 더 감싼다 |

> 둘 중 하나가 옳은 것이 아닙니다. **목록을 여러 개 붙일 때는 패턴 컴프리헨션이 짧고**, 집계와 섞을 때는 `MATCH` + `collect` 가 읽기 좋습니다.

> **담기는 순서는 보장되지 않습니다.** 패턴 컴프리헨션 안에는 `ORDER BY` 를 넣을 수 없으니, 순서가 중요하면 3-3 처럼 `MATCH` 로 잡아 **접기 전에 줄을 세운 뒤** `collect` 하세요.

### 🖐️ 함께 따라하기: 질병 하나에 목록 두 개 붙이기

`asthma`(천식) 한 줄에 목록 두 개를 붙여 보세요.

- `MATCH (d:Disease {name: 'asthma'})` 로 질병 하나만 잡습니다.
- 패턴 컴프리헨션으로 `증상`(`(d)-[:PRESENTS]->(s:Symptom)` 의 `s.name`) 앞 3개와, `치료약수`(`(c:Compound)-[:TREATS]->(d)` 의 개수)를 함께 냅니다.
- 별칭은 `질병`·`증상`·`치료약수` 로 합니다.

확인 기준: `증상` 은 목록 3개짜리이고, `치료약수` 는 **37** 입니다(1-1 상위 5 목록의 asthma 값과 같습니다).

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (d:Disease {name: 'asthma'}) 로 질병 하나만 잡는다
# 2) RETURN 에 [(d)-[:PRESENTS]->(s:Symptom) | s.name][0..3] 을 별칭 증상 으로 둔다
# 3) 같은 RETURN 에 size([(c:Compound)-[:TREATS]->(d) | c]) 를 별칭 치료약수 로 둔다
# 4) 결과를 print 한다

### ✅ 바로 확인 퀴즈

**1)** 패턴 컴프리헨션이 잡을 짝이 하나도 없으면 그 행은 어떻게 되나요?

<details><summary>정답 보기</summary>

**행은 그대로 남고 리스트만 비어 있습니다**(`[]`). `MATCH` 로 이었다면 행 자체가 사라졌을 자리입니다. 그래서 `OPTIONAL MATCH` 대신 쓸 수 있습니다.

</details>

**2)** 패턴 컴프리헨션 안에서 만든 변수(`s`)를 그 뒤 `RETURN` 의 다른 자리에서 쓸 수 있나요?

<details><summary>정답 보기</summary>

쓸 수 없습니다. 대괄호 **안에서만** 사는 변수입니다. 밖에서도 써야 한다면 `MATCH` 로 잡아야 합니다.

</details>

---
# 4. 집계한 뒤 거르기: WITH + WHERE

집계 결과를 조건으로 거르는 자리는 정해져 있습니다. 먼저 그 자리를 익히고(4-1), 같은 일을 **더 짧게 쓰는 다른 길**도 하나 봅니다(4-2).

## 4-1. 집계 결과 조건은 WITH 뒤에

### 왜 필요할까요?
"치료 선택지가 넉넉한 질병만 보고 싶다"처럼 **집계한 값을 조건으로** 거르고 싶을 때가 있습니다. 그런데 `MATCH ... WHERE` 는 **집계하기 전의 낱낱 행**을 거릅니다. 집계 결과를 거르려면 **먼저 집계하고**(WITH) **그 다음에 걸러야**(WHERE) 합니다.

### 문법: 집계는 WITH 로 먼저, 조건은 그 다음
```
MATCH ...
WITH 묶는기준, 집계함수(...) AS 별칭
WHERE 별칭 조건
RETURN ...
```
SQL 을 아는 분이라면 이 자리가 `HAVING` 입니다. Cypher 에는 `HAVING` 이라는 낱말이 없고, `WITH` 뒤의 `WHERE` 가 그 일을 합니다.

<img src="images/where_위치_대비.png" width="760">

`WHERE` 를 `MATCH` 뒤에 두면 **낱낱의 행**을, `WITH` 뒤에 두면 **묶어서 낸 값**을 거릅니다. 같은 낱말인데 놓는 자리가 뜻을 바꿉니다.

In [ ]:
# 치료 약물이 20개 이상인 질병만. 20 은 집계한 뒤에야 알 수 있는 값이라 WITH 뒤에서 건다
# 같은 조건을 MATCH 뒤의 WHERE 에 두면 약물수 라는 이름이 아직 없어 에러가 난다(SQL 의 HAVING 자리)
rows = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WITH d, count(c) AS 약물수
    WHERE 약물수 >= 20
    RETURN d.name AS 질병, 약물수 ORDER BY 약물수 DESC, 질병
""")
for row in rows:
    print(row)

In [ ]:
# 위 목록을 세어 요약한다. 집계한 값에 건 조건이 얼마나 걸러 냈는지 보는 자리다
print("조건에 맞은 질병 수:", len(rows))

질병 77개 중 10개만 남았습니다. 이 목록이 곧 "약이 여럿이라 **고를 수 있는** 병" 이고, 나머지는 선택지가 좁은 병입니다. 집계 뒤 필터는 이렇게 **목록을 쓸 만한 크기로 줄이는** 데 씁니다.

### ✅ 바로 확인 퀴즈

**1)** `MATCH (c)-[:TREATS]->(d) WHERE count(c) >= 20 RETURN d.name` 은 왜 안 될까요?

<details><summary>정답 보기</summary>

`MATCH` 뒤의 `WHERE` 는 아직 집계하지 않은 **낱낱의 행**을 봅니다. 그 자리에서는 `count(c)` 가 존재하지 않습니다. `WITH d, count(c) AS 약물수` 로 먼저 집계한 뒤 걸러야 합니다.

</details>

**2)** `WITH` 뒤에 적지 않은 변수를 다음 절에서 쓰면 어떻게 되나요?

<details><summary>정답 보기</summary>

그 이름을 모른다는 에러가 납니다. `WITH` 는 **적어 준 것만** 다음 단계로 넘깁니다. 뒤에서 쓸 값은 빠짐없이 적어야 합니다.

</details>

---
## 4-2. 짧게 쓰는 다른 길: `COUNT { }` 서브쿼리

### 왜 필요할까요?
4-1 에서 배운 `MATCH` → `WITH` 집계 → `WHERE` 는 **어떤 집계에도 통하는 정석**입니다. 다만 "이 노드에 관계가 몇 개 달렸나" 하나만 알고 싶을 때는 단계를 나누는 것이 번거롭습니다.

그럴 때 쓰는 것이 `COUNT { }` 입니다. **중괄호 안의 패턴에 맞는 행이 몇 개인지**를 그 자리에서 세어 값 하나로 돌려줍니다. 집계 함수가 아니라 **값을 만드는 식**이라 `RETURN`·`WHERE` 어디에나 그냥 쓸 수 있습니다.

### 문법
```
COUNT { (기준)-[:관계]->(:레이블) }
```
안에서는 **바깥의 변수를 그대로** 씁니다(따로 넘겨 줄 필요가 없습니다). 그리고 패턴만 적을 때는 `MATCH` 라는 낱말도 생략합니다.

### 이런 것이 `COUNT` 만 있는 것은 아닙니다

중괄호 안을 들여다보고 **값 하나를 돌려주는** 식이 셋 있습니다. 묶어서 **서브쿼리 식**이라 부릅니다.

| 식 | 돌려주는 것 | 중괄호 안에 |
|---|---|---|
| `EXISTS { }` | 있나 없나(`true`·`false`) | 패턴만 적어도 된다 |
| `COUNT { }` | 몇 개인지(정수) | 패턴만 적어도 된다 |
| `COLLECT { }` | 값들을 모은 리스트 | `MATCH` 와 `RETURN` 을 갖춘 쿼리를 적어야 한다 |

`EXISTS { }` 는 지난 단원에서 이미 썼습니다. 셋 다 바깥 변수를 그대로 쓰고, **행을 늘리지 않고** 값 하나만 돌려준다는 점이 같습니다.

In [ ]:
# 셋을 한 줄에 나란히 낸다. 셋 다 바깥의 d 를 그대로 쓴다
# COLLECT { } 만 중괄호 안에 MATCH 와 RETURN 을 갖춰 적어야 한다
print(run_cypher("""
    MATCH (d:Disease {name: 'migraine'})
    RETURN EXISTS { (:Compound)-[:TREATS]->(d) } AS 치료약있나,
           COUNT { (:Compound)-[:TREATS]->(d) } AS 치료약수,
           COLLECT { MATCH (c:Compound)-[:TREATS]->(d)
                     RETURN c.name ORDER BY c.name LIMIT 3 } AS 치료약맛보기
"""))

In [ ]:
# 질병마다 치료약 수와 완화약 수를 한 줄에 나란히 낸다
# WITH 로 나누는 정석으로 쓰면 관계 종류가 둘이라 단계를 두 번 나눠야 한다.
# COUNT { } 는 행마다 그 자리에서 세므로 MATCH 는 질병 하나뿐이다
rows = run_cypher("""
    MATCH (d:Disease)
    RETURN d.name AS 질병,
           // COUNT { } 안의 패턴은 그 행의 d 를 그대로 쓴다. 화살표 방향만 다르게 두 번 센다
           COUNT { (:Compound)-[:TREATS]->(d) } AS 치료약수,
           COUNT { (:Compound)-[:PALLIATES]->(d) } AS 완화약수
    ORDER BY 치료약수 DESC, 질병 LIMIT 3
""")
for row in rows:
    print(row)

`WHERE` 에도 그대로 쓸 수 있습니다. 4-1 에서 `WITH` 로 나눠 걸렀던 조건을 한 줄로 써 보겠습니다.

In [ ]:
# 같은 물음을 두 방식으로 나란히 푼다: 치료 약물이 20개 이상인 질병은 몇 개인가
# 4-1 의 정석. MATCH -> WITH 로 집계 -> WHERE 로 거르기, 세 단계를 거친다
staged = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WITH d, count(c) AS 약물수
    WHERE 약물수 >= 20
    RETURN count(d) AS 선택지많은질병
""")

# COUNT { } 는 WHERE 안에서 그 자리에 세므로 단계를 나누지 않는다. MATCH 도 질병 하나뿐이다
inline = run_cypher("""
    MATCH (d:Disease)
    WHERE COUNT { (:Compound)-[:TREATS]->(d) } >= 20
    RETURN count(d) AS 선택지많은질병
""")

print("WITH 로 나눠 세면:", staged[0], "· COUNT { } 로 세면:", inline[0])

4-1 에서 `WITH` 로 걸러 낸 개수와 같은 **10** 이 나왔습니다. 같은 답을 두 가지 길로 낸 것입니다.

| | `WITH` 집계 (4-1) | `COUNT { }` (4-2) |
|---|---|---|
| 언제 쓰나 | 집계가 여럿이거나 그 값을 뒤에서 또 쓸 때 | 개수 하나만 필요할 때 |
| 여러 종류를 한 줄에 | 단계를 나눠야 한다 | 나란히 적으면 된다 |
| 집계값에 이름을 붙여 재사용 | `WITH` 로 이름을 붙여 계속 쓴다 | 쓸 때마다 다시 센다 |

> **`WITH` 집계가 기본이고 `COUNT { }` 는 짧게 쓰는 길**입니다. 순서를 바꿔 익히면 집계 파이프라인을 못 짜게 되니, 정석을 먼저 몸에 익히고 이 표현은 "이런 것도 있다" 로 알아 두세요. 뒤 단원의 실무 쿼리에서 자주 만납니다.

### ✅ 바로 확인 퀴즈

**1)** `COUNT { }` 안에서 바깥의 변수 `d` 를 쓰려면 따로 넘겨 줘야 하나요?

<details><summary>정답 보기</summary>

아니요. **바깥 변수를 그대로** 쓸 수 있습니다. 반대로 중괄호 **안에서 만든** 변수는 밖에서 쓸 수 없습니다.

</details>

**2)** `RETURN avg(count(x))` 는 안 되는데 `RETURN avg(COUNT { ... })` 는 왜 될까요?

<details><summary>정답 보기</summary>

`count(x)` 는 **집계 함수**라 집계 함수 안에 겹쳐 쓸 수 없습니다. `COUNT { }` 는 집계가 아니라 **행마다 값을 만드는 식**이라 `avg` 의 재료로 넣을 수 있습니다. 다만 읽기는 `WITH` 로 나눈 쪽이 쉬우니, 단계가 보이게 쓰는 편을 기본으로 두세요.

</details>

---
# 5. 집계 결과를 저장: SET 파생 속성

계산한 값을 그래프에 **붙여 두는** 일입니다. 저장하는 법을 보고(5-1), 저장한 값이 **언제 낡는지**와 정리하는 법을 봅니다(5-2).

## 5-1. 집계값을 노드 속성으로 저장하기

### 왜 필요할까요?
같은 집계를 화면을 볼 때마다 다시 계산하면 낭비입니다. 자주 쓰는 값은 **노드 속성으로 저장**해 두면 다음부터는 계산 없이 바로 읽습니다. 이렇게 **원래 데이터에서 계산해 만들어 붙인 속성**을 파생 속성이라고 합니다.

### 문법: SET / REMOVE
| 표현 | 하는 일 |
|---|---|
| `SET n.속성 = 값` | 속성을 만들거나 덮어쓴다 |
| `REMOVE n.속성` | 속성을 지운다 |

저장하는 값은 숫자만이 아닙니다. 비교식의 결과(참·거짓)도 그대로 넣을 수 있습니다.

<img src="images/set_파생속성_사본.png" width="760">

저장한 값은 **그 순간의 사본**입니다. 원본 관계가 바뀌어도 저장해 둔 수는 그대로이므로, 다시 계산해 넣어 주기 전까지는 낡은 값이 남습니다.

In [ ]:
# SET 으로 drug_count 를 저장한다. 있으면 갈아 끼우므로 다시 실행해도 결과가 같다
run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WITH d, count(c) AS 약물수
    SET d.drug_count = 약물수
""")

# 관계 대신 속성만 읽어 확인한다. 치료약 없는 59개 질병은 속성이 없어 IS NOT NULL 로 거른다
rows = run_cypher("""
    MATCH (d:Disease) WHERE d.drug_count IS NOT NULL
    RETURN d.name AS 질병, d.drug_count AS 약물수
    ORDER BY 약물수 DESC, 질병 LIMIT 3
""")
for row in rows:
    print(row)

> `WHERE d.drug_count IS NOT NULL` 이 필요한 이유: 치료약이 없는 59개 질병은 패턴에 안 맞아 `SET` 이 지나가지 않았습니다. 그 노드들에는 이 속성이 아예 없습니다. **파생 속성은 있을 수도 없을 수도 있다**는 것을 늘 염두에 두세요.

In [ ]:
# 저장하는 값이 꼭 숫자일 필요는 없다: 비교식의 결과(참/거짓)를 그대로 넣을 수 있다
run_cypher("""
    MATCH (d:Disease) WHERE d.drug_count IS NOT NULL
    SET d.has_many_options = (d.drug_count >= 20)
""")
# 참/거짓 속성은 비교식 없이 WHERE 뒤에 이름만 적어도 조건이 된다
rows = run_cypher("""
    MATCH (d:Disease) WHERE d.has_many_options
    RETURN count(d) AS 선택지많은질병
""")
print(rows)

---
## 5-2. 저장한 값은 낡는다: REMOVE 로 정리하기

### 왜 필요할까요?
파생 속성은 **그때 계산한 값의 사본**입니다. 원본 관계가 바뀌어도 저장해 둔 수는 그대로라, 다시 계산해 덮어쓰기 전까지는 낡은 값이 남습니다. 그래서 잠깐 쓰려고 만든 속성은 **쓰고 나서 지웁니다.** 남겨 두면 뒤 실습이 낡은 값을 읽습니다.

In [ ]:
# 잠깐 만들어 본 속성은 REMOVE 로 지운다(뒤 실습이 이 속성을 기대하지 않도록)
run_cypher("MATCH (d:Disease) REMOVE d.drug_count, d.has_many_options")
print(run_cypher("""
    MATCH (d:Disease) WHERE d.drug_count IS NOT NULL
    RETURN count(d) AS 남은속성
"""))

### 🖐️ 함께 따라하기: 증상에 등장 질병 수 저장

질병-증상 축에서 파생 속성을 만들어 보세요.

1. 증상마다 등장 질병 수를 세어 `s.disease_count` 로 저장합니다.
2. 저장한 속성만 읽어 상위 3개를 출력합니다(별칭 `증상`·`질병수`).
3. 마지막에 `REMOVE` 로 지웁니다.

확인 기준: 1위는 `Edema`(49)입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (d:Disease)-[:PRESENTS]->(s:Symptom) 뒤 WITH s, count(d) AS 질병수 로 집계
# 2) SET s.disease_count = 질병수  로 저장
# 3) 관계를 타지 않고 s.disease_count 만 읽어 상위 3개를 출력
# 4) REMOVE s.disease_count 로 정리하고, 남은 개수가 0 인지 확인

### ✅ 바로 확인 퀴즈

**1)** 새 `TREATS` 관계 하나를 더 넣으면, 저장해 둔 `drug_count` 도 저절로 1 늘어날까요?

<details><summary>정답 보기</summary>

아니요. `SET` 이 저장한 것은 **그때 계산한 값의 사본**입니다. 관계가 바뀌면 다시 계산해 덮어써야 합니다. 파생 속성은 빠르지만 낡을 수 있다는 것이 대가입니다.

</details>

**2)** `SET d.drug_count = 약물수` 를 이미 값이 있는 노드에 다시 실행하면 어떻게 되나요?

<details><summary>정답 보기</summary>

덮어씁니다. `SET` 은 없으면 만들고 있으면 갈아 끼웁니다. 그래서 여러 번 실행해도 결과가 같습니다(멱등).

</details>

---
# 6. 조건별로 값 만들기: CASE

`CASE` 는 세 걸음으로 봅니다. 범위를 나누는 **조건형**(6-1), 값이 몇 개로 정해져 있을 때 쓰는 **단순형**(6-2), 그리고 그 `CASE` 를 **집계 함수 안에 넣어** 행마다 점수를 매기는 법(6-3)입니다.

## 6-1. 조건형 CASE: 구간 라벨 만들기

### 왜 필요할까요?
숫자를 그대로 보면 읽기 어렵습니다. "치료약 68개"보다 "선택지 많음" 이 한눈에 들어옵니다. `CASE` 는 값을 **조건에 따라 다른 값으로 바꿔** 줍니다. 파이썬의 `if/elif/else` 와 같은 자리입니다.

### 문법: CASE 두 가지 형태
```
CASE WHEN 조건1 THEN 값1 WHEN 조건2 THEN 값2 ELSE 값3 END     조건형(범위·복합 조건)
CASE 대상 WHEN 값A THEN 값1 WHEN 값B THEN 값2 ELSE 값3 END    단순형(값이 같은지만)
```
만들어 낸 라벨은 **그냥 값**이라, 그룹핑 키로도 쓰고 `collect` 로 모을 수도 있습니다.

<img src="images/case_조건순서.png" width="760">

조건은 **위에서 아래로** 훑어 처음 맞는 가지에서 멈춥니다. 그래서 **넓은 조건을 위에 두면 그 안에 들어가는 좁은 가지는 영영 걸리지 않습니다**(`5 이상` 을 위에 두면 `20 이상` 가지가 그렇습니다). 다만 어느 조건에도 안 걸린 값은 그대로 `ELSE` 로 갑니다. `ELSE` 가 없으면 그 값이 `null` 이 됩니다.

In [ ]:
# 조건형 CASE: 치료 약물 수를 구간 라벨로 바꾼다. 좁은 조건(20 이상)을 먼저 쓴다
# 순서를 뒤집어 >= 5 를 위에 두면 20 이상인 값도 거기서 먼저 걸려 '선택지 많음' 이 한 번도 안 나온다
rows = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WITH d, count(c) AS 약물수
    RETURN d.name AS 질병, 약물수,
           CASE WHEN 약물수 >= 20 THEN '선택지 많음'
                WHEN 약물수 >= 5  THEN '보통'
                ELSE '선택지 적음' END AS 약물수구간
    ORDER BY 약물수 DESC, 질병 LIMIT 5
""")
for row in rows:
    print(row)

In [ ]:
# CASE 결과를 WITH 로 넘겨 이름을 붙여야 그 다음에서 묶는 기준으로 쓸 수 있다
# 두 번째 WITH 에 d 를 함께 적은 이유: 뒤에서 count(d)·collect(d.name) 에 쓰려면 넘겨 줘야 한다
rows = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WITH d, count(c) AS 약물수
    WITH CASE WHEN 약물수 >= 20 THEN '선택지 많음'
              WHEN 약물수 >= 5  THEN '보통'
              ELSE '선택지 적음' END AS 약물수구간, d
    RETURN 약물수구간, count(d) AS 질병수, collect(d.name)[0..3] AS 질병맛보기
    ORDER BY 질병수 DESC
""")
for row in rows:
    print(row)

---
## 6-2. 단순형 CASE: 값이 딱 정해진 몇 개일 때

비교 대상이 하나이고 그 값이 무엇인지만 따질 때는 `CASE` 뒤에 대상을 한 번만 적습니다.

이 그래프에서 그런 자리가 바로 **`TREATS` 와 `PALLIATES`** 입니다. 둘 다 "약과 병을 잇는 관계"지만 뜻이 다릅니다. `type(r)` 로 관계 종류를 꺼내 라벨을 붙여 보겠습니다.

In [ ]:
# type(r) 은 관계의 종류 이름('TREATS' 같은 문자열)을 돌려준다
# TREATS|PALLIATES 로 두 관계를 한 번에 잡고, CASE 로 다시 갈라 세는 것이 이 셀의 요점이다
rows = run_cypher("""
    MATCH (c:Compound)-[r:TREATS|PALLIATES]->(:Disease {name: 'migraine'})
    RETURN CASE type(r) WHEN 'TREATS' THEN '병 자체 치료'
                        ELSE '증상 완화' END AS 하는일,
           count(c) AS 약물수 ORDER BY 약물수 DESC
""")
for row in rows:
    print(row)

편두통에 쓰이는 약은 **증상 완화 24개, 병 자체 치료 7개** 입니다. 약이 훨씬 많은 쪽은 "아픔을 덜어 주는" 쪽입니다. 이 둘을 뭉뚱그려 "편두통 치료약 31개" 라고 세면 **의학적으로 틀린 말**이 됩니다.

그래프 전체로도 `TREATS` 는 755건, `PALLIATES` 는 390건으로 나뉘어 있습니다. 데이터가 이렇게 갈라 두었다면, 집계할 때도 갈라서 세는 것이 옳습니다.

---
## 6-3. 집계 함수 안에서 계산하기
지금까지는 `count(c)`·`sum(질병수)` 처럼 **값 하나**를 그대로 집계했습니다. 그런데 집계 함수 안에는 **식**을 통째로 넣을 수 있습니다. 행마다 먼저 계산한 뒤, 그 결과를 합치는 것입니다.

```text
sum(수량 * 단가)                       행마다 곱한 뒤 합친다(매출 합계가 이 모양이다)
sum(CASE WHEN 조건 THEN 2 ELSE 1 END)   행마다 점수를 매긴 뒤 합친다
```

약물이 질병에 하는 일을 **가중해서** 세어 보겠습니다. 앞 절에서 봤듯 `TREATS`(병 자체 치료)와 `PALLIATES`(증상 완화)는 무게가 다릅니다. 치료는 **2점**, 완화는 **1점**으로 두고 약물마다 합칩니다.

In [ ]:
# sum 안에 CASE 를 넣어 행마다 점수를 매긴 뒤 합친다(그냥 count 로 세면 둘을 같게 세게 된다)
# 가중점수 옆에 질병수(count)를 나란히 두면, 같은 질병 수라도 점수가 갈리는 것이 보인다
rows = run_cypher("""
    MATCH (c:Compound)-[r:TREATS|PALLIATES]->(d:Disease)
    RETURN c.name AS 약물,
           sum(CASE type(r) WHEN 'TREATS' THEN 2 ELSE 1 END) AS 가중점수,
           count(d) AS 질병수
    ORDER BY 가중점수 DESC, 약물 LIMIT 5
""")
for row in rows:
    print(row)

1위 **Methotrexate** 은 질병 19건에 가중점수 38 입니다. 질병 수만 세면 치료와 완화가 같은 무게로 섞이지만, 집계 안에서 먼저 점수를 매기면 **데이터가 갈라 둔 차이가 결과에 남습니다.**

> 이 모양은 실무에서 자주 나옵니다. 매출을 낼 때 `sum(수량 * 단가)` 로 쓰는 것도 같은 문법입니다. **행마다 계산하고 나서 합친다**는 순서를 기억하세요.

### 🖐️ 함께 따라하기: 증상 등장 빈도를 구간 라벨로

질병-증상 축에서 조건형 `CASE` 를 연습합니다.

증상마다 등장 질병 수를 센 뒤, **20개 이상은 `'아주 흔함'`, 5개 이상은 `'흔함'`, 나머지는 `'드묾'`** 으로 라벨을 붙이고, 라벨마다 증상이 몇 개인지 세어 많은 순으로 출력하세요. 별칭은 `구간`·`증상수` 로 합니다.

확인 기준: 세 구간이 모두 나오고, 합이 등장 질병이 있는 증상 수와 같습니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (d:Disease)-[:PRESENTS]->(s:Symptom) 뒤 WITH s, count(d) AS 질병수 로 1차 집계
# 2) WITH CASE WHEN ... END AS 질병수구간  으로 라벨을 만들어 넘긴다 (좁은 조건 20 이상을 먼저)
# 3) RETURN 질병수구간, count(*) AS 증상수  로 라벨별 개수를 세고 많은 순으로 정렬
# 4) 세 구간의 증상수 합을 함께 print 해 빠진 것이 없는지 확인

### ✅ 바로 확인 퀴즈

**1)** 조건을 `WHEN 약물수 >= 5 THEN '보통' WHEN 약물수 >= 20 THEN '선택지 많음'` 순으로 뒤집으면 어떻게 되나요?

<details><summary>정답 보기</summary>

`'선택지 많음'` 이 **한 번도 나오지 않습니다.** 20 이상인 값도 위의 `>= 5` 에 먼저 걸려 거기서 멈추기 때문입니다. 조건형 `CASE` 는 **좁은 조건을 위에** 두어야 합니다.

</details>

**2)** `ELSE` 를 빼면 어떤 값이 나오나요?

<details><summary>정답 보기</summary>

아무 조건에도 걸리지 않은 행은 `null` 이 됩니다. 그 `null` 도 하나의 그룹으로 묶여 세어지므로, 결과에 `None` 이라는 구간이 생깁니다.

</details>

**3)** "이 병에 쓸 수 있는 약이 몇 개인가"를 물었을 때 `TREATS` 와 `PALLIATES` 를 합쳐 세도 되나요?

<details><summary>정답 보기</summary>

물음에 따라 다릅니다. "쓸 수 있는 약"이라면 합쳐도 되지만, **"이 병을 치료하는 약"이라면 안 됩니다.** `PALLIATES` 는 증상을 눅이는 것이지 병을 다스리는 것이 아닙니다. 데이터가 갈라 둔 것을 집계에서 합칠 때는 **그 합이 무슨 뜻인지 말로 설명할 수 있어야** 합니다.

</details>

---
# 7. 내장 함수: 외우지 말고 찾아 쓰기

오늘까지 쓴 함수는 스무 개 남짓입니다. 그런데 이 데이터베이스에는 훨씬 많이 들어 있습니다. 먼저 **내 데이터베이스에 무엇이 있는지 직접 세어 보고**(7-1), 갈래별로 **하나씩 써 보고**(7-2), 마지막으로 **공식 문서에서 찾는 법**을 익힙니다(7-3).

## 7-1. 내 데이터베이스에 함수가 몇 개나 있나

### 왜 필요할까요?
"이런 걸 해 주는 함수가 있을까?" 는 실무에서 하루에 몇 번씩 하는 질문입니다. 그때 검색창에 묻기 전에 **데이터베이스에 직접 물어볼 수 있습니다.** 내 판·내 버전에서 정말 쓸 수 있는 것만 답해 주니 더 정확합니다.

### 문법: SHOW FUNCTIONS
| 표현 | 하는 일 |
|---|---|
| `SHOW FUNCTIONS` | 이 데이터베이스가 아는 함수를 전부 내놓는다 |
| `SHOW FUNCTIONS YIELD name, category` | 필요한 열만 골라 받는다 |
| `... WHERE name STARTS WITH 'to'` | 이름으로 좁힌다 |

`SHOW INDEXES`·`SHOW CONSTRAINTS` 와 같은 모양입니다. `YIELD` 로 열을 고르고 그 뒤에 `WHERE`·`RETURN`·`ORDER BY` 를 평소처럼 이어 씁니다.

In [ ]:
# 이 데이터베이스가 아는 함수가 모두 몇 개인지 세어 본다
# SHOW 로 시작하는 문장도 YIELD 뒤에 RETURN·집계를 평소처럼 이어 쓸 수 있다
print(run_cypher("SHOW FUNCTIONS YIELD name RETURN count(*) AS 함수수"))

In [ ]:
# 갈래(category)별로 몇 개인지. category 가 빈 문자열인 것은 내부용이라 걸러 낸다
for row in run_cypher("""
    SHOW FUNCTIONS YIELD name, category WHERE category <> ''
    RETURN category AS 갈래, count(*) AS 함수수
    ORDER BY 함수수 DESC, 갈래
"""):
    print(row)

함수가 **449개**입니다(이 판 기준. 버전마다 다릅니다). 다 외우는 사람은 없습니다. 대신 **갈래를 알면** 찾을 수 있습니다. 오늘까지 우리가 쓴 것을 갈래에 얹어 보면 이렇습니다.

| 갈래 | 무엇을 하나 | 오늘까지 쓴 것 |
|---|---|---|
| `Aggregating` | 여러 행을 한 줄로 요약 | `count`·`sum`·`avg`·`min`·`max`·`collect`·`percentileCont`·`stDev` |
| `Scalar` | 값 하나를 다른 값 하나로 | `size`·`coalesce`·`toFloat` |
| `String` | 문자열 다듬기 | (7-2 에서 씁니다) |
| `Numeric` | 수치 계산 | `round` |
| `List` | 리스트 다루기 | `keys`·`labels`·`range` |
| `Predicate` | 참·거짓 판정 | (7-2 에서 씁니다) |
| `Graph` | 그래프 구조 | `type`·`nodes`·`relationships`(지난 시간) |
| `Temporal` | 날짜·시간 | 이 데이터에는 시간 값이 없어 쓰지 않습니다 |
| `Vector`·`Spatial`·`Trigonometric` | 벡터·공간·삼각함수 | 뒤 단원과 이 과정 범위 밖 |

> 갈래 이름이 곧 **공식 문서의 페이지 이름**입니다. 7-3 에서 그 링크를 정리합니다.

In [ ]:
# 이름으로 좁혀 찾기: 'to' 로 시작하는 Scalar 함수는 자료형을 바꾸는 함수들이다
# 무엇이 있는지 모를 때 이렇게 훑으면 이름만 보고도 쓸 것을 고를 수 있다
print(run_cypher("""
    SHOW FUNCTIONS YIELD name, category
    WHERE name STARTS WITH 'to' AND category = 'Scalar'
    RETURN collect(name) AS 이름들
"""))


> `...OrNull` 이 붙은 짝이 눈에 띕니다. `toInteger('abc')` 는 에러가 나지만 `toIntegerOrNull('abc')` 는 `null` 을 줍니다. **바꿀 수 없는 값이 섞여 있을 수 있는 데이터**를 다룰 때 뒤쪽을 씁니다. 이름 짓는 규칙만 알아도 문서를 안 열고 고를 수 있습니다.

---
## 7-2. 갈래별로 하나씩 써 보기

### 왜 필요할까요?
함수는 읽어서 익히는 것보다 **한 번 돌려 보는 것**이 빠릅니다. 갈래마다 대표적인 것만 골라 한 줄씩 돌려 보겠습니다. 여기서는 그래프를 쓰지 않고 `WITH` 로 값을 하나 만들어 시험합니다(문법을 시험할 때 쓰는 흔한 요령입니다).

In [ ]:
# 문자열 다듬기: 앞뒤 공백을 떼고, 대문자로 바꾸고, 쪼개고, 갈아 끼운다
# WITH 로 값 하나를 만들어 두면 그래프 없이도 함수를 시험할 수 있다
print(run_cypher("""
    WITH '  Warfarin, Aspirin  ' AS 원본
    RETURN trim(원본) AS 공백제거, toUpper(trim(원본)) AS 대문자,
           split(trim(원본), ', ') AS 쪼개기,
           replace(trim(원본), 'Aspirin', 'Ibuprofen') AS 바꾸기
"""))


In [ ]:
# 문자열 잘라내기: 앞에서 몇 자, 뒤에서 몇 자, 가운데 몇 자
# substring(문자열, 시작, 길이) 의 시작은 0 부터 센다. size 는 글자 수다
print(run_cypher("""
    WITH 'Acetylsalicylic acid' AS 이름
    RETURN left(이름, 5) AS 앞5, right(이름, 4) AS 뒤4,
           substring(이름, 6, 9) AS 잘라내기, size(이름) AS 글자수
"""))


In [ ]:
# 수치: 절댓값·올림·내림·부호. round 는 자릿수를 둘째 인자로 받는다
# ceil·floor 는 정수처럼 보여도 FLOAT 로 돌아온다(10.0 · 9.0)
print(run_cypher("""
    RETURN abs(-7) AS 절댓값, ceil(9.1) AS 올림, floor(9.9) AS 내림,
           sign(-3) AS 부호, round(9.815, 2) AS 반올림
"""))


In [ ]:
# 리스트 함수. head·last 는 값 하나를 주고, tail·reverse 는 리스트를 준다
# tail 은 '끝' 이 아니라 첫 것을 뺀 나머지 전부다(약물들[1..] 과 같다). 끝 하나는 last 다
# 3절의 자리 번호(목록[0])와 하는 일이 같다. 이름이 붙어 있어 읽기 좋을 때 쓴다
print(run_cypher("""
    WITH ['Carvedilol', 'Clonidine', 'Captopril', 'Amiloride'] AS 약물들
    RETURN head(약물들) AS 첫값, last(약물들) AS 끝값,
           tail(약물들) AS 첫것뺀나머지, 약물들[1..] AS 자리번호로같은것,
           reverse(약물들) AS 뒤집기, range(1, 5, 2) AS 등차수열
"""))


In [ ]:
# 스칼라: 빠진 값 메우기와 자료형 바꾸기. valueType 은 그 값이 무엇인지 알려 준다
# coalesce 는 앞에서부터 처음으로 null 이 아닌 값을 준다(다음 단원 적재에서도 씁니다)
print(run_cypher("""
    RETURN coalesce(null, null, '기본값') AS 첫값, toInteger('42') AS 정수로,
           toString(9.81) AS 문자열로, valueType(9.81) AS 자료형,
           isEmpty([]) AS 비었나
"""))


In [ ]:
# 서술(참/거짓): 리스트 전체를 한 조건으로 판정한다. WHERE 에 그대로 넣어 쓴다
# single 은 '조건에 맞는 것이 딱 하나' 일 때만 참이다(any 와 다르다)
print(run_cypher("""
    WITH [12, 7, 3] AS 수들
    RETURN all(x IN 수들 WHERE x > 0) AS 모두양수,
           any(x IN 수들 WHERE x > 10) AS 하나라도십초과,
           none(x IN 수들 WHERE x < 0) AS 음수없나,
           single(x IN 수들 WHERE x > 10) AS 딱하나만
"""))


In [ ]:
# 리스트를 하나의 값으로 접기: reduce(누적변수 = 초깃값, x IN 목록 | 식)
# 집계 함수가 행을 접는 것과 달리, reduce 는 한 행 안의 리스트를 접는다
print(run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease)
    WITH d, count(c) AS 약물수 ORDER BY 약물수 DESC LIMIT 5
    WITH collect(약물수) AS 상위5
    RETURN 상위5, reduce(합 = 0, x IN 상위5 | 합 + x) AS 누적합
"""))


> **`sum` 과 `reduce` 는 접는 대상이 다릅니다.** `sum(약물수)` 는 **여러 행**을 한 줄로 접고, `reduce` 는 **한 행 안의 리스트**를 값 하나로 접습니다. 위 셀에서 `collect` 로 리스트를 만든 뒤에야 `reduce` 를 쓴 이유가 그것입니다.

### 🖐️ 함께 따라하기: 함수를 그래프 위에서 조합하기

지금까지는 `WITH` 로 만든 값에 함수를 써 봤습니다. 이번에는 **그래프에서 꺼낸 값**에 씁니다.

`migraine`(편두통)을 치료하는 약물의 이름을 가나다순으로 정렬한 뒤,

1. 각 이름의 **앞 네 글자**만 모아 앞 4개를 내세요(별칭 `앞네글자`).
2. 각 이름의 **글자 수**를 모아 앞 4개를 내세요(별칭 `글자수`).

`collect` 는 담기는 순서를 보장하지 않으니 **접기 전에 `ORDER BY` 로 줄을 세우세요**(3-3 에서 한 그대로입니다).

확인 기준: `앞네글자` 는 `['Amit', 'Gaba', 'Prop', 'Timo']`, `글자수` 는 `[13, 10, 11, 7]` 입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (c:Compound)-[:TREATS]->(:Disease {name: 'migraine'}) 로 치료약을 잡는다
# 2) WITH c.name AS 이름 ORDER BY 이름  으로 접기 전에 줄을 세운다
# 3) RETURN 에 collect(left(이름, 4))[0..4] 를 별칭 앞네글자 로 둔다
# 4) 같은 RETURN 에 collect(size(이름))[0..4] 를 별칭 글자수 로 둔다

### ✅ 바로 확인 퀴즈

**1)** `toInteger('abc')` 와 `toIntegerOrNull('abc')` 는 어떻게 다른가요?

<details><summary>정답 보기</summary>

앞은 **에러**가 나고 뒤는 **`null`** 을 돌려줍니다. CSV 처럼 숫자가 아닌 값이 섞여 들어올 수 있는 데이터를 다룰 때는 뒤쪽을 써서 그 줄만 `null` 로 두고 넘어갑니다.

</details>

**2)** `sum` 과 `reduce` 는 무엇이 다른가요?

<details><summary>정답 보기</summary>

**접는 대상**이 다릅니다. `sum` 은 **여러 행**을 한 줄로 접는 집계 함수이고, `reduce` 는 **한 행 안의 리스트**를 값 하나로 접습니다. 그래서 `reduce` 를 쓰려면 먼저 `collect` 로 리스트를 만들어야 합니다.

</details>

**3)** "이 데이터베이스에서 쓸 수 있는 문자열 함수가 무엇인지" 알고 싶습니다. 무엇을 하나요?

<details><summary>정답 보기</summary>

`SHOW FUNCTIONS YIELD name, category WHERE category = 'String' RETURN name` 을 돌립니다. 검색창보다 정확한 것은, **내 판·내 버전에 실제로 있는 것만** 답해 주기 때문입니다.

</details>

---
## 7-3. 공식 문서에서 찾는 법

### 왜 필요할까요?
`SHOW FUNCTIONS` 는 **이름**을 알려 주지만 **어떻게 쓰는지**는 알려 주지 않습니다. 인자가 몇 개인지, 어떤 자료형을 받는지, 빠진 값을 만나면 어떻게 하는지는 공식 문서에 있습니다. 갈래 이름이 곧 문서 페이지라 찾기 쉽습니다.

### 갈래별 공식 문서
| 갈래 | 무엇이 있나 | 공식 문서 |
|---|---|---|
| 함수 전체 목차 | 갈래를 한눈에 | <https://neo4j.com/docs/cypher-manual/current/functions/> |
| Aggregating | `count`·`sum`·`avg`·`collect`·`percentileCont`·`stDev` | <https://neo4j.com/docs/cypher-manual/current/functions/aggregating/> |
| String | `trim`·`toUpper`·`split`·`replace`·`substring`·`left`·`right` | <https://neo4j.com/docs/cypher-manual/current/functions/string/> |
| Numeric | `abs`·`ceil`·`floor`·`round`·`sign`·`rand` | <https://neo4j.com/docs/cypher-manual/current/functions/mathematical-numeric/> |
| List | `head`·`last`·`tail`·`range`·`reverse`·`reduce`·`keys`·`labels` | <https://neo4j.com/docs/cypher-manual/current/functions/list/> |
| Scalar | `size`·`coalesce`·`toInteger`·`toFloat`·`toString`·`valueType`·`properties` | <https://neo4j.com/docs/cypher-manual/current/functions/scalar/> |
| Predicate | `all`·`any`·`none`·`single`·`isEmpty`·`exists` | <https://neo4j.com/docs/cypher-manual/current/functions/predicate/> |
| Temporal | `date`·`datetime`·`duration` | <https://neo4j.com/docs/cypher-manual/current/functions/temporal/date/> |

> `current` 자리에 버전을 넣으면 그 버전 문서로 갑니다. 지금 쓰는 판이 무엇인지는 `CALL dbms.components()` 로 확인할 수 있습니다. **버전이 다르면 없는 함수도 있으니**, 문서를 열기 전에 `SHOW FUNCTIONS` 로 내 데이터베이스에 있는지 먼저 보는 편이 빠릅니다.

### 찾는 순서 세 걸음

1. **갈래를 정한다.** 문자열을 다듬나(String), 값을 바꾸나(Scalar), 리스트를 다루나(List), 참·거짓을 묻나(Predicate).
2. **`SHOW FUNCTIONS` 로 이름을 훑는다.** `WHERE category = 'String'` 으로 좁히면 스무 줄 남짓이라 눈으로 읽힙니다.
3. **공식 문서에서 그 함수만 읽는다.** 인자 순서와 **빠진 값을 만났을 때**를 꼭 확인합니다(2-2 에서 본 대로 그 자리가 조용히 틀리는 자리입니다).

### ✅ 바로 확인 퀴즈

**1)** 문서 URL 의 `current` 는 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

**최신 판 문서**입니다. 그 자리에 버전을 넣으면 그 버전 문서로 갑니다. 내 데이터베이스 판과 문서 판이 다르면 없는 함수를 읽고 있을 수 있으니, `SHOW FUNCTIONS` 로 먼저 확인하는 것이 안전합니다.

</details>

**2)** 처음 보는 함수를 문서에서 읽을 때 **반드시 확인해야 할 것** 하나는 무엇인가요?

<details><summary>정답 보기</summary>

**빠진 값(`null`)을 만나면 어떻게 하는가.** 2-2 에서 본 대로 그 자리가 에러 없이 답만 틀리는 자리입니다. 문서의 각 함수 설명에 `null` 처리가 한 줄로 적혀 있습니다.

</details>

---
## 🚀 응용 클론코딩: 약효분류별 요약(2단계 집계)

오늘 배운 것을 한 쿼리에 모읍니다. **약효분류**(작용 방식이 같은 약물 묶음)마다 소속 약물이 몇 개이고, 그 약물들이 평균 몇 개의 유전자에 결합하는지를 냅니다.

이 질문이 어려운 이유는 **집계가 두 번** 필요하기 때문입니다.

1. 약물마다 표적 유전자 수를 센다(1차 집계).
2. 그 수들을 약효분류마다 다시 평균 낸다(2차 집계).

한 번에 쓰려고 하면 `avg(count(...))` 가 되어 실행되지 않습니다. `WITH` 로 단계를 나눠야 합니다.

<img src="images/2단_with_파이프라인.png" width="760">

1차 집계 결과에 이름을 붙여 다음 단계로 넘기고, 그 값을 다시 집계합니다. `WITH` 에 적지 않은 값은 뒤에서 쓸 수 없습니다.

**요구사항**

- 약효분류와 그 분류에 든 약물을 잇습니다(전체 1,029건).
- 약물마다 표적 유전자를 이어 붙입니다. **표적이 하나도 없는 약물도 빠지면 안 됩니다.**
- 먼저 약물마다 표적 수를 셉니다(1차 집계).
- 그 수들을 분류마다 다시 모아, 소속 약물 수와 평균 표적 수를 냅니다(2차 집계).
- 약물 수 내림차순, 동점이면 분류 이름순으로 상위 5개만 냅니다.
- 결과 칸 이름은 `약효분류`·`약물수`·`평균표적수` 로 하고, 평균은 소수 둘째 자리까지 다듬습니다.

확인 기준: 1위는 `Corticosteroid Hormone Receptor Agonists`(약물수 22)입니다. 약효분류는 모두 345개이고, 소속 약물 수의 평균은 2.98개입니다.

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) MATCH (p:PharmacologicClass)-[:INCLUDES]->(c:Compound)
# 2) OPTIONAL MATCH (c)-[:BINDS]->(g:Gene)   표적이 없는 약물도 남긴다
# 3) WITH p, c, count(g) AS 표적수           1차: 약물마다 표적 수
# 4) RETURN 에서 count(c)·round(avg(표적수), 2)  2차: 약효분류마다 다시 집계
# 5) ORDER BY 약물수 DESC, 약효분류 LIMIT 5

> 결과를 읽을 때 주의할 것이 하나 있습니다. 소속 약물이 **1개뿐인 분류**도 많습니다(전체 345개 중 최솟값이 1). 그런 분류의 "평균 표적 수"는 사실 한 약물의 값일 뿐입니다. **평균을 낼 때는 몇 개를 평균 낸 것인지 함께 봐야** 합니다.

---
## 이번 강의 정리

| 하고 싶은 일 | 쓰는 것 |
|---|---|
| 행 수를 센다 | `count(*)` |
| 값이 있는 것만 센다 | `count(x)` |
| 빠진 값을 다른 값으로 채운다 | `coalesce(x, 0)` (집계 전에 행마다 걸린다) |
| 서로 다른 것을 센다 | `count(DISTINCT x)` |
| 항목별로 묶어 센다 | RETURN 에 **비집계 항**을 함께 둔다 |
| 집계한 수를 다시 집계한다 | `WITH` 로 넘기고 `min`·`max`·`avg`·`sum` |
| 목록으로 모은다 | `collect(x)`, 길이는 `size(...)` |
| 목록을 다시 행으로 편다 | `UNWIND 리스트 AS x` |
| 집계 결과를 조건으로 거른다 | `WITH ... WHERE ...` (SQL 의 HAVING) |
| 집계 결과를 저장한다 | `SET n.속성 = 값`, 지울 때 `REMOVE` |
| 조건별 라벨을 만든다 | `CASE WHEN ... THEN ... ELSE ... END` |
| 집계하기 전에 행마다 계산한다 | `sum(수량 * 단가)`, `sum(CASE ... END)` |
| 중복 없이 모은다 | `collect(DISTINCT x)` |
| 분포를 본다 | `percentileCont(x, 0.5)`(중앙값)·`percentileCont(x, 0.9)`·`stDev(x)` |
| 비율을 만든다 | `toFloat(a) / b` (정수끼리 나누면 버림이다) |
| 리스트를 조건으로 거른다 | `[x IN 목록 WHERE 조건 \| 바꿀식]` |
| 패턴을 바로 리스트로 받는다 | `[(a)-[:R]->(b) \| b.name]` (짝이 없으면 빈 리스트) |
| 개수 하나만 짧게 센다 | `COUNT { (a)-[:R]->(:B) }` |
| 문자열을 다듬는다 | `trim`·`toUpper`·`split`·`replace`·`substring`·`left`·`right` |
| 수치를 다듬는다 | `abs`·`ceil`·`floor`·`sign`·`round(값, 자릿수)` |
| 리스트를 다룬다 | `head`·`last`·`tail`·`reverse`·`range`·`reduce` |
| 값을 바꾸거나 메운다 | `toInteger`·`toString`·`valueType`·`coalesce`·`isEmpty` |
| 리스트 전체를 판정한다 | `all`·`any`·`none`·`single` |
| 쓸 수 있는 함수를 찾는다 | `SHOW FUNCTIONS YIELD name, category WHERE ...` |

**오늘 데이터에서 배운 것**

- 질병 136개 중 치료약이 연결된 것은 77개, 나머지 59개는 없습니다. `count(*)` 와 `count(x)` 의 차이가 그 수를 드러냅니다.
- 그래프는 경로가 갈라져 같은 노드를 여러 번 셉니다. **경로가 갈라지는 자리에서 개수를 물으면 `DISTINCT`** 를 붙입니다(한 홉짜리 패턴은 갈라지지 않아 없어도 같습니다).
- `TREATS` 와 `PALLIATES` 는 다른 말입니다. 데이터가 갈라 둔 것을 집계에서 합치면 뜻이 달라집니다.
- 평균 하나로는 분포를 알 수 없습니다. 평균 9.81 옆의 중앙값은 7 이었습니다. `percentileCont` 로 중앙값을, `CASE` 로 구간을 만들어 함께 보세요.
- **집계는 빠진 값을 조용히 건너뜁니다.** 평균 옆에는 `count(*)` 를 함께 두어 몇 개를 평균 낸 것인지 늘 보이게 하세요.
- **정수끼리 나누면 소수점이 버려집니다.** 비율을 만들 때는 한쪽을 `toFloat` 로 감싸세요.
- 이 데이터베이스에는 함수가 **449개**(이 판 기준) 있습니다. 외우는 것이 아니라 **갈래를 알고 `SHOW FUNCTIONS` 와 공식 문서로 찾는 것**이 실력입니다(7절).

## ⏭️ 예고: 다음 시간

오늘 맨 위에서 그냥 실행하고 넘어간 적재 셀에는 아직 설명하지 않은 줄이 둘 있었습니다. `CREATE CONSTRAINT` 로 시작하는 줄과, `MATCH` 에 `:Compound` 같은 **레이블을 꼬박꼬박 적은** 것입니다. 다음 시간에는 그 두 가지가 왜 필요한지를 **실행계획으로 직접 재어** 확인합니다. 레이블 하나를 빼먹으면 같은 적재가 얼마나 느려지는지도 봅니다.

그리고 오늘 만든 집계를 **추천·랭킹**으로 잇습니다. "이 병에 쓸 약을 무엇으로 줄 세울 것인가", "이 약과 표적을 많이 공유하는 다른 약은 무엇인가" 같은 질문입니다.

수고하셨습니다!